In [3]:
import sys
print(sys.executable)

c:\Users\DELL\api_data_pipeline\.venv\Scripts\python.exe


In [4]:
import dotenv
import requests

print("Environment is ready!")

Environment is ready!


In [6]:
import requests
import sys
!{sys.executable} -m pip install pandas
import pandas as pd

print("Requests:", requests.__version__)
print("Pandas:", pd.__version__)

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.8 MB 3.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.8 MB 4.2 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.8 MB 4.3 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.8 MB 4.4 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.8 MB 4.5 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.8 MB 4.5 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/9.8 MB 4.6 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.8 MB 4.6 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 4.5 MB/s  0:00:02
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -- -----------------------------

In [19]:
weather_url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.06263,
    "longitude": 31.24967,
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "relative_humidity_2m_mean",
        "wind_speed_10m_mean"
    ],
    "timezone": "auto",
    "past_days": 7
}

response = requests.get(
    weather_url,
    params=params,
    timeout=10
)

print("Status:", response.status_code)

Status: 200


In [20]:
data = response.json()

print(data.keys())
print(data["daily"].keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
dict_keys(['time', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'relative_humidity_2m_mean', 'wind_speed_10m_mean'])


In [21]:
from pathlib import Path
import json

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

raw_file = raw_dir / "cairo_weather_raw.json"

with open(raw_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved: {raw_file}")

Saved: data\raw\cairo_weather_raw.json


In [27]:
cities = [
    "Riyadh",
    "Salalah",
    "Amman",
    "Istanbul",
    "London",
    "Munich",
    "Zurich",
    "Cairo",
    "Khartoum"
]

print(cities)
print("Number of cities:", len(cities))

['Riyadh', 'Salalah', 'Amman', 'Istanbul', 'London', 'Munich', 'Zurich', 'Cairo', 'Khartoum']
Number of cities: 9


In [28]:
import requests
import pandas as pd

geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

locations = []

for city in cities:
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        geocoding_url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    result = response.json()

    if "results" not in result or not result["results"]:
        print(f"Not found: {city}")
        continue

    location = result["results"][0]

    locations.append({
        "city": city,
        "country": location["country"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "timezone": location["timezone"]
    })

locations_df = pd.DataFrame(locations)

print(locations_df)

       city              country  latitude  longitude         timezone
0    Riyadh         Saudi Arabia  24.68773   46.72185      Asia/Riyadh
1   Salalah                 Oman  17.01505   54.09237      Asia/Muscat
2     Amman               Jordan  31.95522   35.94503       Asia/Amman
3  Istanbul  Republic of Türkiye  41.01384   28.94966  Europe/Istanbul
4    London       United Kingdom  51.50853   -0.12574    Europe/London
5    Munich              Germany  48.13743   11.57549    Europe/Berlin
6    Zurich          Switzerland  47.36667    8.55000    Europe/Zurich
7     Cairo                Egypt  30.06263   31.24967     Africa/Cairo
8  Khartoum                Sudan  15.55177   32.53241  Africa/Khartoum


In [29]:

weather_raw = {}

for _, location in locations_df.iterrows():

    city = location["city"]
    latitude = location["latitude"]
    longitude = location["longitude"]

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "wind_speed_10m_mean"
        ],
        "timezone": "auto",
        "past_days": 7
    }

    response = requests.get(
        weather_url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    weather_raw[city] = response.json()

    print(f"{city}: {response.status_code}")

Riyadh: 200
Salalah: 200
Amman: 200
Istanbul: 200
London: 200
Munich: 200
Zurich: 200
Cairo: 200
Khartoum: 200


In [30]:
from pathlib import Path
import json

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

for city, weather_data in weather_raw.items():
    file_path = raw_dir / f"{city.lower()}_weather_raw.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(weather_data, f, ensure_ascii=False, indent=2)

    print(f"Saved: {file_path}")

Saved: data\raw\riyadh_weather_raw.json
Saved: data\raw\salalah_weather_raw.json
Saved: data\raw\amman_weather_raw.json
Saved: data\raw\istanbul_weather_raw.json
Saved: data\raw\london_weather_raw.json
Saved: data\raw\munich_weather_raw.json
Saved: data\raw\zurich_weather_raw.json
Saved: data\raw\cairo_weather_raw.json
Saved: data\raw\khartoum_weather_raw.json


In [31]:
all_data = []

for city, weather_data in weather_raw.items():

    daily = weather_data["daily"]

    city_df = pd.DataFrame(daily)

    city_df["city"] = city

    all_data.append(city_df)

df = pd.concat(all_data, ignore_index=True)

print(df.shape)
print(df.head())

(126, 8)
         time  temperature_2m_mean  temperature_2m_max  temperature_2m_min  \
0  2026-09-06                 35.3                39.5                29.6   
1  2026-09-07                 37.0                42.1                31.3   
2  2026-09-08                 37.9                43.1                32.4   
3  2026-09-09                 38.6                43.2                33.9   
4  2026-09-10                 38.6                42.7                33.4   

   precipitation_sum  relative_humidity_2m_mean  wind_speed_10m_mean    city  
0                0.0                         21                  5.5  Riyadh  
1                0.0                         14                  9.6  Riyadh  
2                0.0                         14                  8.6  Riyadh  
3                0.0                         13                  6.4  Riyadh  
4                0.0                         13                  6.1  Riyadh  


In [32]:
print(df.groupby("city")["time"].count())

city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: time, dtype: int64


In [33]:
df["time"] = pd.to_datetime(df["time"])

print(df.dtypes)

time                         datetime64[us]
temperature_2m_mean                 float64
temperature_2m_max                  float64
temperature_2m_min                  float64
precipitation_sum                   float64
relative_humidity_2m_mean             int64
wind_speed_10m_mean                 float64
city                                    str
dtype: object


In [34]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicates:", df.duplicated().sum())

Shape: (126, 8)

Missing values:
time                         0
temperature_2m_mean          0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
relative_humidity_2m_mean    0
wind_speed_10m_mean          0
city                         0
dtype: int64

Duplicates: 0


In [35]:
print("Temperature:")
print("Min:", df["temperature_2m_min"].min())
print("Max:", df["temperature_2m_max"].max())

print("\nHumidity:")
print("Min:", df["relative_humidity_2m_mean"].min())
print("Max:", df["relative_humidity_2m_mean"].max())

print("\nPrecipitation:")
print("Min:", df["precipitation_sum"].min())

print("\nWind Speed:")
print("Min:", df["wind_speed_10m_mean"].min())

Temperature:
Min: 9.2
Max: 44.6

Humidity:
Min: 8
Max: 92

Precipitation:
Min: 0.0

Wind Speed:
Min: 2.7


In [36]:
print("Temperature:")
print("Min:", df["temperature_2m_min"].min())
print("Max:", df["temperature_2m_max"].max())

print("\nHumidity:")
print("Min:", df["relative_humidity_2m_mean"].min())
print("Max:", df["relative_humidity_2m_mean"].max())

print("\nPrecipitation:")
print("Min:", df["precipitation_sum"].min())

print("\nWind Speed:")
print("Min:", df["wind_speed_10m_mean"].min())

Temperature:
Min: 9.2
Max: 44.6

Humidity:
Min: 8
Max: 92

Precipitation:
Min: 0.0

Wind Speed:
Min: 2.7


In [37]:
print(df.groupby("city")["time"].count())

city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: time, dtype: int64


In [38]:
print("City-Date duplicates:",
      df.duplicated(subset=["city", "time"]).sum())

City-Date duplicates: 0


In [39]:
df = df.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temp_avg",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "precipitation",
    "relative_humidity_2m_mean": "humidity",
    "wind_speed_10m_mean": "wind_speed"
})

print(df.columns.tolist())

['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city']


In [40]:
print(df.columns.tolist())

['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city']


In [41]:
dim_city = locations_df[
    ["city", "country", "latitude", "longitude"]
].copy()

dim_city.insert(0, "city_key", range(1, len(dim_city) + 1))

print(dim_city)

   city_key      city              country  latitude  longitude
0         1    Riyadh         Saudi Arabia  24.68773   46.72185
1         2   Salalah                 Oman  17.01505   54.09237
2         3     Amman               Jordan  31.95522   35.94503
3         4  Istanbul  Republic of Türkiye  41.01384   28.94966
4         5    London       United Kingdom  51.50853   -0.12574
5         6    Munich              Germany  48.13743   11.57549
6         7    Zurich          Switzerland  47.36667    8.55000
7         8     Cairo                Egypt  30.06263   31.24967
8         9  Khartoum                Sudan  15.55177   32.53241


In [42]:
dim_date = df[["date"]].drop_duplicates().sort_values("date").copy()

dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()

print(dim_date)

         date  date_key  year  month month_name
56 2026-09-05  20260905  2026      9  September
0  2026-09-06  20260906  2026      9  September
1  2026-09-07  20260907  2026      9  September
2  2026-09-08  20260908  2026      9  September
3  2026-09-09  20260909  2026      9  September
4  2026-09-10  20260910  2026      9  September
5  2026-09-11  20260911  2026      9  September
6  2026-09-12  20260912  2026      9  September
7  2026-09-13  20260913  2026      9  September
8  2026-09-14  20260914  2026      9  September
9  2026-09-15  20260915  2026      9  September
10 2026-09-16  20260916  2026      9  September
11 2026-09-17  20260917  2026      9  September
12 2026-09-18  20260918  2026      9  September
13 2026-09-19  20260919  2026      9  September


In [43]:
print("DataFrame rows:", len(df))
print("Unique dates:", df["date"].nunique())
print("Expected rows:", len(dim_city) * dim_date["date"].nunique())

print("\nRows per city:")
print(df.groupby("city")["date"].count())

DataFrame rows: 126
Unique dates: 15
Expected rows: 135

Rows per city:
city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: date, dtype: int64


In [44]:
print("DataFrame rows:", len(df))
print("Unique dates:", df["date"].nunique())
print("Expected rows:", len(dim_city) * dim_date["date"].nunique())

print("\nRows per city:")

print(df.groupby("city")["date"].count())

DataFrame rows: 126
Unique dates: 15
Expected rows: 135

Rows per city:
city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: date, dtype: int64


In [45]:
expected_dates = set(df["date"].unique())

for city in df["city"].unique():
    city_dates = set(df.loc[df["city"] == city, "date"])
    missing_dates = sorted(expected_dates - city_dates)

    if missing_dates:
        print(city, "missing:", missing_dates)

Riyadh missing: [Timestamp('2026-09-05 00:00:00')]
Salalah missing: [Timestamp('2026-09-05 00:00:00')]
Amman missing: [Timestamp('2026-09-05 00:00:00')]
Istanbul missing: [Timestamp('2026-09-05 00:00:00')]
London missing: [Timestamp('2026-09-19 00:00:00')]
Munich missing: [Timestamp('2026-09-05 00:00:00')]
Zurich missing: [Timestamp('2026-09-05 00:00:00')]
Cairo missing: [Timestamp('2026-09-05 00:00:00')]
Khartoum missing: [Timestamp('2026-09-05 00:00:00')]


In [ ]:
from pathlib import Path
import json

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

for city, weather_data in weather_raw.items():
    file_path = raw_dir / f"{city.lower()}_weather_raw.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(weather_data, f, ensure_ascii=False, indent=2)

    print(f"Saved: {file_path}")

Saved: data\raw\riyadh_weather_raw.json
Saved: data\raw\salalah_weather_raw.json
Saved: data\raw\amman_weather_raw.json
Saved: data\raw\istanbul_weather_raw.json
Saved: data\raw\london_weather_raw.json
Saved: data\raw\munich_weather_raw.json
Saved: data\raw\zurich_weather_raw.json
Saved: data\raw\cairo_weather_raw.json
Saved: data\raw\khartoum_weather_raw.json


In [47]:
from pathlib import Path

batch_dir = Path("data/batches/2025-01")

raw_batch_dir = batch_dir / "raw"
checkpoint_dir = batch_dir / "checkpoint"

raw_batch_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Batch directory:", batch_dir)
print("Raw directory:", raw_batch_dir)
print("Checkpoint directory:", checkpoint_dir)

Batch directory: data\batches\2025-01
Raw directory: data\batches\2025-01\raw
Checkpoint directory: data\batches\2025-01\checkpoint


In [51]:
import json
import requests
import pandas as pd
from pathlib import Path
from calendar import monthrange

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

YEAR = 2025

HISTORICAL_WEATHER_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)

DAILY_VARIABLES = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "relative_humidity_2m_mean",
    "wind_speed_10m_mean"
]

BASE_BATCH_DIR = Path("data/batches")


# ============================================================
# 2. FUNCTION: EXTRACT ONE MONTH
# ============================================================

def extract_month(year, month, locations_df):

    month_str = f"{year}-{month:02d}"

    start_date = f"{year}-{month:02d}-01"

    last_day = monthrange(year, month)[1]

    end_date = f"{year}-{month:02d}-{last_day:02d}"

    print("\n")
    print("=" * 60)
    print(f"PROCESSING BATCH: {month_str}")
    print(f"Date range: {start_date} → {end_date}")
    print("=" * 60)

    # --------------------------------------------------------
    # Batch directories
    # --------------------------------------------------------

    batch_dir = BASE_BATCH_DIR / month_str

    raw_dir = batch_dir / "raw"

    checkpoint_dir = batch_dir / "checkpoint"

    raw_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_file = (
        checkpoint_dir / f"weather_{month_str}.csv"
    )

    # --------------------------------------------------------
    # Resume protection
    # --------------------------------------------------------

    if checkpoint_file.exists():

        print(
            f"CHECKPOINT ALREADY EXISTS: "
            f"{checkpoint_file}"
        )

        existing_df = pd.read_csv(
            checkpoint_file
        )

        print(
            f"Skipping API extraction."
            f" Existing rows: {len(existing_df)}"
        )

        return existing_df

    # --------------------------------------------------------
    # Extract each city
    # --------------------------------------------------------

    all_data = []

    for _, location in locations_df.iterrows():

        city = location["city"]

        latitude = location["latitude"]

        longitude = location["longitude"]

        raw_file = (
            raw_dir /
            f"{city.lower()}_weather_raw.json"
        )

        print(f"\nRequesting: {city}")

        params = {
            "latitude": latitude,
            "longitude": longitude,
            "daily": DAILY_VARIABLES,
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date
        }

        try:

            response = requests.get(
                HISTORICAL_WEATHER_URL,
                params=params,
                timeout=30
            )

            print(
                f"Status: {response.status_code}"
            )

            response.raise_for_status()

            weather_data = response.json()

            # ------------------------------------------------
            # Save RAW immediately
            # ------------------------------------------------

            with open(
                raw_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    weather_data,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            # ------------------------------------------------
            # Convert RAW → DataFrame
            # ------------------------------------------------

            daily = weather_data["daily"]

            city_df = pd.DataFrame(daily)

            city_df["city"] = city

            all_data.append(city_df)

            print(
                f"Saved: {raw_file} | "
                f"Days: {len(city_df)}"
            )

        except Exception as e:

            print(
                f"ERROR in {city}: {e}"
            )

            print(
                "Stopping this batch."
            )

            raise

    # ========================================================
    # 3. COMBINE CITIES
    # ========================================================

    batch_df = pd.concat(
        all_data,
        ignore_index=True
    )

    # ========================================================
    # 4. RENAME COLUMNS
    # ========================================================

    batch_df = batch_df.rename(
        columns={
            "time": "date",
            "temperature_2m_mean": "temp_avg",
            "temperature_2m_max": "temp_max",
            "temperature_2m_min": "temp_min",
            "precipitation_sum": "precipitation",
            "relative_humidity_2m_mean": "humidity",
            "wind_speed_10m_mean": "wind_speed"
        }
    )

    # ========================================================
    # 5. DATE TYPE
    # ========================================================

    batch_df["date"] = pd.to_datetime(
        batch_df["date"]
    )

    # ========================================================
    # 6. COLUMN ORDER
    # ========================================================

    batch_df = batch_df[
        [
            "date",
            "city",
            "temp_avg",
            "temp_max",
            "temp_min",
            "precipitation",
            "humidity",
            "wind_speed"
        ]
    ]

    # ========================================================
    # 7. SAVE CSV CHECKPOINT
    # ========================================================

    batch_df.to_csv(
        checkpoint_file,
        index=False
    )

    # ========================================================
    # 8. VERIFY BATCH
    # ========================================================

    expected_rows = (
        len(locations_df) *
        last_day
    )

    actual_rows = len(batch_df)

    print("\n")
    print("-" * 60)
    print(f"BATCH COMPLETED: {month_str}")
    print("-" * 60)

    print(
        f"Expected rows: {expected_rows}"
    )

    print(
        f"Actual rows:   {actual_rows}"
    )

    print(
        f"Cities:        "
        f"{batch_df['city'].nunique()}"
    )

    print(
        f"Unique dates:  "
        f"{batch_df['date'].nunique()}"
    )

    print(
        f"CSV checkpoint: "
        f"{checkpoint_file}"
    )

    # --------------------------------------------------------
    # City row verification
    # --------------------------------------------------------

    print("\nRows per city:")

    print(
        batch_df
        .groupby("city")["date"]
        .count()
    )

    # --------------------------------------------------------
    # Basic validation
    # --------------------------------------------------------

    if actual_rows != expected_rows:

        raise ValueError(
            f"Row count mismatch "
            f"for {month_str}: "
            f"expected {expected_rows}, "
            f"got {actual_rows}"
        )

    print(
        "\nSTATUS: SUCCESS"
    )

    return batch_df


# ============================================================
# 9. RUN ALL 12 MONTHS
# ============================================================

monthly_results = {}

for month in range(1, 13):

    month_df = extract_month(
        YEAR,
        month,
        locations_df
    )

    monthly_results[
        f"{YEAR}-{month:02d}"
    ] = month_df


# ============================================================
# 10. FINAL YEAR SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("2025 FULL YEAR EXTRACTION COMPLETED")
print("=" * 60)

total_rows = 0

for month_str, month_df in monthly_results.items():

    rows = len(month_df)

    total_rows += rows

    print(
        f"{month_str}: {rows} rows"
    )

print("-" * 60)

print(
    f"TOTAL ROWS: {total_rows}"
)

expected_year_rows = (
    len(locations_df) * 365
)

print(
    f"EXPECTED:   {expected_year_rows}"
)

print("=" * 60)

if total_rows == expected_year_rows:

    print(
        "YEAR STATUS: SUCCESS"
    )

else:

    print(
        "YEAR STATUS: CHECK REQUIRED"
    )



PROCESSING BATCH: 2025-01
Date range: 2025-01-01 → 2025-01-31
CHECKPOINT ALREADY EXISTS: data\batches\2025-01\checkpoint\weather_2025-01.csv
Skipping API extraction. Existing rows: 279


PROCESSING BATCH: 2025-02
Date range: 2025-02-01 → 2025-02-28

Requesting: Riyadh
Status: 200
Saved: data\batches\2025-02\raw\riyadh_weather_raw.json | Days: 28

Requesting: Salalah
Status: 200
Saved: data\batches\2025-02\raw\salalah_weather_raw.json | Days: 28

Requesting: Amman
Status: 200
Saved: data\batches\2025-02\raw\amman_weather_raw.json | Days: 28

Requesting: Istanbul
Status: 200
Saved: data\batches\2025-02\raw\istanbul_weather_raw.json | Days: 28

Requesting: London
Status: 200
Saved: data\batches\2025-02\raw\london_weather_raw.json | Days: 28

Requesting: Munich
Status: 200
Saved: data\batches\2025-02\raw\munich_weather_raw.json | Days: 28

Requesting: Zurich
Status: 200
Saved: data\batches\2025-02\raw\zurich_weather_raw.json | Days: 28

Requesting: Cairo
Status: 200
Saved: data\batches\2

In [53]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

YEAR = 2025

BASE_BATCH_DIR = Path("data/batches")

FINAL_DATA_DIR = Path("data/final")
FINAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

final_csv = FINAL_DATA_DIR / "weather_2025.csv"


# ============================================================
# 2. CANONICAL CITY NAMES
# ============================================================

city_name_map = {
    "riyadh": "Riyadh",
    "salalah": "Salalah",
    "amman": "Amman",
    "istanbul": "Istanbul",
    "london": "London",
    "munich": "Munich",
    "zurich": "Zurich",
    "cairo": "Cairo",
    "khartoum": "Khartoum"
}

expected_cities = set(
    city_name_map.values()
)


# ============================================================
# 3. LOAD ALL 12 CHECKPOINTS
# ============================================================

monthly_data = []

print("=" * 60)
print("LOADING 2025 MONTHLY CHECKPOINTS")
print("=" * 60)

for month in range(1, 13):

    month_str = f"{YEAR}-{month:02d}"

    checkpoint_file = (
        BASE_BATCH_DIR
        / month_str
        / "checkpoint"
        / f"weather_{month_str}.csv"
    )

    print(f"\nLoading: {month_str}")

    if not checkpoint_file.exists():

        raise FileNotFoundError(
            f"Missing checkpoint: {checkpoint_file}"
        )

    month_df = pd.read_csv(
        checkpoint_file
    )

    month_df["date"] = pd.to_datetime(
        month_df["date"]
    )

    print(
        f"Rows loaded: {len(month_df)}"
    )

    # ========================================================
    # 4. STANDARDIZE CITY NAMES
    # ========================================================

    month_df["city"] = (
        month_df["city"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(city_name_map)
    )

    # Check for unmapped cities
    if month_df["city"].isna().any():

        bad_values = (
            month_df.loc[
                month_df["city"].isna()
            ]
        )

        raise ValueError(
            f"Unknown city names found in "
            f"{month_str}:\n{bad_values}"
        )

    monthly_data.append(
        month_df
    )


# ============================================================
# 5. COMBINE ALL MONTHS
# ============================================================

df_2025 = pd.concat(
    monthly_data,
    ignore_index=True
)

print("\n")
print("=" * 60)
print("ALL MONTHS COMBINED")
print("=" * 60)

print(
    "Rows:",
    len(df_2025)
)

print(
    "Columns:",
    len(df_2025.columns)
)


# ============================================================
# 6. EXPECTED COLUMNS
# ============================================================

expected_columns = [
    "date",
    "city",
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

if list(df_2025.columns) != expected_columns:

    raise ValueError(
        "Column structure does not match expected schema."
    )

print(
    "Column structure: PASS"
)


# ============================================================
# 7. YEAR CHECK
# ============================================================

years = sorted(
    df_2025["date"]
    .dt.year
    .unique()
)

print("\nYears found:", years)

if years != [YEAR]:

    raise ValueError(
        f"Unexpected years found: {years}"
    )

print(
    "Year check: PASS"
)


# ============================================================
# 8. DATE RANGE
# ============================================================

min_date = df_2025["date"].min()

max_date = df_2025["date"].max()

print("\nDate range:")
print(
    min_date,
    "→",
    max_date
)

if (
    min_date != pd.Timestamp("2025-01-01")
    or
    max_date != pd.Timestamp("2025-12-31")
):

    raise ValueError(
        "Incorrect date range."
    )

print(
    "Date range: PASS"
)


# ============================================================
# 9. TOTAL ROW COUNT
# ============================================================

expected_rows = 9 * 365

actual_rows = len(df_2025)

print("\nRow count:")
print(
    "Expected:",
    expected_rows
)

print(
    "Actual:",
    actual_rows
)

if actual_rows != expected_rows:

    raise ValueError(
        f"Expected {expected_rows} rows "
        f"but found {actual_rows}."
    )

print(
    "Row count: PASS"
)


# ============================================================
# 10. CITY CHECK
# ============================================================

actual_cities = set(
    df_2025["city"].unique()
)

print("\nCities found:")
print(
    sorted(actual_cities)
)

print(
    "Number of cities:",
    len(actual_cities)
)

if actual_cities != expected_cities:

    print(
        "Missing cities:",
        expected_cities - actual_cities
    )

    print(
        "Unexpected cities:",
        actual_cities - expected_cities
    )

    raise ValueError(
        "City validation failed."
    )

print(
    "City check: PASS"
)


# ============================================================
# 11. ROWS PER CITY
# ============================================================

rows_per_city = (
    df_2025
    .groupby("city")["date"]
    .count()
    .sort_index()
)

print("\nRows per city:")
print(
    rows_per_city
)

if not (rows_per_city == 365).all():

    raise ValueError(
        "At least one city does not have 365 rows."
    )

print(
    "Rows per city: PASS"
)


# ============================================================
# 12. UNIQUE DATES
# ============================================================

unique_dates = (
    df_2025["date"].nunique()
)

print("\nUnique dates:", unique_dates)

if unique_dates != 365:

    raise ValueError(
        f"Expected 365 unique dates, "
        f"found {unique_dates}."
    )

print(
    "Unique dates: PASS"
)


# ============================================================
# 13. CITY-DATE GRAIN
# ============================================================

duplicate_city_date = (
    df_2025
    .duplicated(
        subset=["city", "date"]
    )
    .sum()
)

print(
    "\nDuplicate city-date combinations:",
    duplicate_city_date
)

if duplicate_city_date != 0:

    raise ValueError(
        "Duplicate city-date records found."
    )

print(
    "City-date grain: PASS"
)


# ============================================================
# 14. MISSING VALUES
# ============================================================

missing_values = (
    df_2025
    .isna()
    .sum()
)

print("\nMissing values:")
print(
    missing_values
)

if missing_values.sum() != 0:

    raise ValueError(
        "Missing values detected."
    )

print(
    "Missing values: PASS"
)


# ============================================================
# 15. FULL DUPLICATES
# ============================================================

full_duplicates = (
    df_2025
    .duplicated()
    .sum()
)

print(
    "\nFull duplicate rows:",
    full_duplicates
)

if full_duplicates != 0:

    raise ValueError(
        "Full duplicate rows found."
    )

print(
    "Full duplicates: PASS"
)


# ============================================================
# 16. TEMPERATURE LOGIC
# ============================================================

invalid_temperature = (
    (df_2025["temp_min"] > df_2025["temp_avg"])
    |
    (df_2025["temp_avg"] > df_2025["temp_max"])
)

invalid_temp_count = (
    invalid_temperature.sum()
)

print(
    "\nInvalid temperature records:",
    invalid_temp_count
)

if invalid_temp_count != 0:

    raise ValueError(
        "Invalid temperature relationships found."
    )

print(
    "Temperature logic: PASS"
)


# ============================================================
# 17. PRECIPITATION
# ============================================================

negative_precipitation = (
    df_2025["precipitation"] < 0
).sum()

print(
    "\nNegative precipitation:",
    negative_precipitation
)

if negative_precipitation != 0:

    raise ValueError(
        "Negative precipitation detected."
    )

print(
    "Precipitation: PASS"
)


# ============================================================
# 18. HUMIDITY
# ============================================================

invalid_humidity = (
    (df_2025["humidity"] < 0)
    |
    (df_2025["humidity"] > 100)
).sum()

print(
    "\nInvalid humidity:",
    invalid_humidity
)

if invalid_humidity != 0:

    raise ValueError(
        "Invalid humidity values detected."
    )

print(
    "Humidity: PASS"
)


# ============================================================
# 19. WIND SPEED
# ============================================================

negative_wind = (
    df_2025["wind_speed"] < 0
).sum()

print(
    "\nNegative wind speed:",
    negative_wind
)

if negative_wind != 0:

    raise ValueError(
        "Negative wind speed detected."
    )

print(
    "Wind speed: PASS"
)


# ============================================================
# 20. COMPLETE CITY-DATE COVERAGE
# ============================================================

expected_dates = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="D"
)

coverage_errors = []

print("\n")
print("=" * 60)
print("CITY-DATE COVERAGE")
print("=" * 60)

for city in sorted(expected_cities):

    city_dates = (
        df_2025.loc[
            df_2025["city"] == city,
            "date"
        ]
        .drop_duplicates()
        .sort_values()
    )

    missing_dates = (
        expected_dates
        .difference(city_dates)
    )

    extra_dates = (
        city_dates[
            ~city_dates.isin(
                expected_dates
            )
        ]
    )

    print(
        f"{city}: "
        f"{len(city_dates)} dates | "
        f"Missing: {len(missing_dates)}"
    )

    if len(missing_dates) > 0:

        coverage_errors.append(
            (
                city,
                "Missing",
                list(missing_dates)
            )
        )

    if len(extra_dates) > 0:

        coverage_errors.append(
            (
                city,
                "Unexpected",
                list(extra_dates)
            )
        )


if coverage_errors:

    print("\nCoverage errors:")

    for error in coverage_errors:
        print(error)

    raise ValueError(
        "City-date coverage validation failed."
    )

print(
    "\nComplete city-date coverage: PASS"
)


# ============================================================
# 21. SORT FINAL DATASET
# ============================================================

df_2025 = (
    df_2025
    .sort_values(
        ["date", "city"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 22. SAVE FINAL DATASET
# ============================================================

df_2025.to_csv(
    final_csv,
    index=False
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("FINAL 2025 DATASET CREATED")
print("=" * 60)

print(
    "File:",
    final_csv
)

print(
    "Rows:",
    len(df_2025)
)

print(
    "Columns:",
    len(df_2025.columns)
)

print(
    "Cities:",
    df_2025["city"].nunique()
)

print(
    "Dates:",
    df_2025["date"].nunique()
)

print("\n")
print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print("Rows:                 PASS")
print("Columns:              PASS")
print("Year:                 PASS")
print("Date range:           PASS")
print("Cities:               PASS")
print("Rows per city:        PASS")
print("Unique dates:         PASS")
print("City-date grain:      PASS")
print("Missing values:       PASS")
print("Full duplicates:      PASS")
print("Temperature logic:    PASS")
print("Precipitation:        PASS")
print("Humidity:             PASS")
print("Wind speed:           PASS")
print("Date coverage:        PASS")

print("=" * 60)
print("2025 DATASET STATUS: SUCCESS")
print("=" * 60)

LOADING 2025 MONTHLY CHECKPOINTS

Loading: 2025-01
Rows loaded: 279

Loading: 2025-02
Rows loaded: 252

Loading: 2025-03
Rows loaded: 279

Loading: 2025-04
Rows loaded: 270

Loading: 2025-05
Rows loaded: 279

Loading: 2025-06
Rows loaded: 270

Loading: 2025-07
Rows loaded: 279

Loading: 2025-08
Rows loaded: 279

Loading: 2025-09
Rows loaded: 270

Loading: 2025-10
Rows loaded: 279

Loading: 2025-11
Rows loaded: 270

Loading: 2025-12
Rows loaded: 279


ALL MONTHS COMBINED
Rows: 3285
Columns: 8
Column structure: PASS

Years found: [np.int32(2025)]
Year check: PASS

Date range:
2025-01-01 00:00:00 → 2025-12-31 00:00:00
Date range: PASS

Row count:
Expected: 3285
Actual: 3285
Row count: PASS

Cities found:
['Amman', 'Cairo', 'Istanbul', 'Khartoum', 'London', 'Munich', 'Riyadh', 'Salalah', 'Zurich']
Number of cities: 9
City check: PASS

Rows per city:
city
Amman       365
Cairo       365
Istanbul    365
Khartoum    365
London      365
Munich      365
Riyadh      365
Salalah     365
Zurich   

In [56]:
# ============================================================
# EDA - PART 1
# Statistical & Distribution Overview
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Basic dataset information
# ------------------------------------------------------------

print("=" * 60)
print("EDA - DATASET OVERVIEW")
print("=" * 60)

print("Shape:", df_2025.shape)

print("\nData types:")
print(df_2025.dtypes)

print("\nMemory usage:")
print(df_2025.memory_usage(deep=True).sum(), "bytes")


# ------------------------------------------------------------
# 2. Statistical summary
# ------------------------------------------------------------

numeric_columns = [
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

print("\n")
print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)

print(
    df_2025[numeric_columns].describe()
)


# ------------------------------------------------------------
# 3. City-level statistics
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CITY-LEVEL TEMPERATURE STATISTICS")
print("=" * 60)

city_temperature = (
    df_2025
    .groupby("city")
    .agg(
        avg_temperature=("temp_avg", "mean"),
        max_temperature=("temp_max", "max"),
        min_temperature=("temp_min", "min")
    )
    .sort_values(
        "avg_temperature",
        ascending=False
    )
)

print(city_temperature)


# ------------------------------------------------------------
# 4. City-level precipitation
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CITY-LEVEL PRECIPITATION")
print("=" * 60)

city_precipitation = (
    df_2025
    .groupby("city")
    .agg(
        total_precipitation=("precipitation", "sum"),
        average_daily_precipitation=("precipitation", "mean"),
        rainy_days=(
            "precipitation",
            lambda x: (x > 0).sum()
        )
    )
    .sort_values(
        "total_precipitation",
        ascending=False
    )
)

print(city_precipitation)


# ------------------------------------------------------------
# 5. City-level humidity and wind
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CITY-LEVEL HUMIDITY & WIND")
print("=" * 60)

city_environment = (
    df_2025
    .groupby("city")
    .agg(
        average_humidity=("humidity", "mean"),
        max_humidity=("humidity", "max"),
        average_wind_speed=("wind_speed", "mean"),
        max_wind_speed=("wind_speed", "max")
    )
    .sort_values(
        "average_humidity",
        ascending=False
    )
)

print(city_environment)


# ------------------------------------------------------------
# 6. Correlation matrix
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CORRELATION MATRIX")
print("=" * 60)

correlation = (
    df_2025[numeric_columns]
    .corr()
)

print(correlation)


# ------------------------------------------------------------
# 7. Extreme temperature records
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("TOP 10 HOTTEST DAYS")
print("=" * 60)

hottest_days = (
    df_2025
    .sort_values(
        "temp_max",
        ascending=False
    )
    [
        [
            "date",
            "city",
            "temp_avg",
            "temp_max",
            "temp_min"
        ]
    ]
    .head(10)
)

print(hottest_days)


print("\n")
print("=" * 60)
print("TOP 10 COLDEST DAYS")
print("=" * 60)

coldest_days = (
    df_2025
    .sort_values(
        "temp_min",
        ascending=True
    )
    [
        [
            "date",
            "city",
            "temp_avg",
            "temp_max",
            "temp_min"
        ]
    ]
    .head(10)
)

print(coldest_days)


# ------------------------------------------------------------
# 8. Highest precipitation days
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("TOP 10 RAINIEST DAYS")
print("=" * 60)

rainiest_days = (
    df_2025
    .sort_values(
        "precipitation",
        ascending=False
    )
    [
        [
            "date",
            "city",
            "precipitation"
        ]
    ]
    .head(10)
)

print(rainiest_days)


# ------------------------------------------------------------
# 9. Highest wind days
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("TOP 10 WINDIEST DAYS")
print("=" * 60)

windiest_days = (
    df_2025
    .sort_values(
        "wind_speed",
        ascending=False
    )
    [
        [
            "date",
            "city",
            "wind_speed"
        ]
    ]
    .head(10)
)

print(windiest_days)

print("\n")
print("=" * 60)
print("EDA PART 1 COMPLETED")
print("=" * 60)

EDA - DATASET OVERVIEW
Shape: (3285, 8)

Data types:
date             datetime64[us]
city                        str
temp_avg                float64
temp_max                float64
temp_min                float64
precipitation           float64
humidity                  int64
wind_speed              float64
dtype: object

Memory usage:
365862 bytes


STATISTICAL SUMMARY
          temp_avg     temp_max     temp_min  precipitation     humidity  \
count  3285.000000  3285.000000  3285.000000    3285.000000  3285.000000   
mean     19.707549    24.514581    15.251446       1.159756    56.631659   
std      10.004478    10.967675     9.020915       3.576252    25.148901   
min      -4.500000    -2.200000    -8.100000       0.000000     6.000000   
25%      11.900000    16.100000     8.500000       0.000000    37.000000   
50%      20.600000    25.900000    15.600000       0.000000    61.000000   
75%      27.200000    32.800000    22.700000       0.200000    78.000000   
max      40.400000 

In [57]:
# ============================================================
# EDA - PART 1B
# Compact Summary
# ============================================================

print("=" * 60)
print("COMPACT EDA SUMMARY")
print("=" * 60)

print("\nDataset:")
print("Rows:", len(df_2025))
print("Cities:", df_2025["city"].nunique())
print("Date range:", df_2025["date"].min(), "to", df_2025["date"].max())

print("\n" + "=" * 60)
print("NUMERIC SUMMARY")
print("=" * 60)

summary = df_2025[
    [
        "temp_avg",
        "temp_max",
        "temp_min",
        "precipitation",
        "humidity",
        "wind_speed"
    ]
].agg(["min", "mean", "max"])

print(summary.round(2))

print("\n" + "=" * 60)
print("AVERAGE TEMPERATURE BY CITY")
print("=" * 60)

city_temp = (
    df_2025
    .groupby("city")["temp_avg"]
    .mean()
    .sort_values(ascending=False)
)

print(city_temp.round(2))

print("\n" + "=" * 60)
print("TOTAL PRECIPITATION BY CITY")
print("=" * 60)

city_rain = (
    df_2025
    .groupby("city")["precipitation"]
    .sum()
    .sort_values(ascending=False)
)

print(city_rain.round(2))

print("\n" + "=" * 60)
print("AVERAGE HUMIDITY BY CITY")
print("=" * 60)

city_humidity = (
    df_2025
    .groupby("city")["humidity"]
    .mean()
    .sort_values(ascending=False)
)

print(city_humidity.round(2))

print("\n" + "=" * 60)
print("AVERAGE WIND SPEED BY CITY")
print("=" * 60)

city_wind = (
    df_2025
    .groupby("city")["wind_speed"]
    .mean()
    .sort_values(ascending=False)
)

print(city_wind.round(2))

print("\n" + "=" * 60)
print("COMPACT SUMMARY COMPLETED")
print("=" * 60)

COMPACT EDA SUMMARY

Dataset:
Rows: 3285
Cities: 9
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00

NUMERIC SUMMARY
      temp_avg  temp_max  temp_min  precipitation  humidity  wind_speed
min      -4.50     -2.20     -8.10           0.00      6.00        1.00
mean     19.71     24.51     15.25           1.16     56.63        8.75
max      40.40     46.20     36.90          41.80     99.00       34.60

AVERAGE TEMPERATURE BY CITY
city
Khartoum    30.90
Riyadh      28.07
Salalah     26.09
Cairo       24.03
Amman       19.28
Istanbul    16.13
London      12.42
Zurich      10.38
Munich      10.08
Name: temp_avg, dtype: float64

TOTAL PRECIPITATION BY CITY
city
Zurich      1267.5
Munich       840.7
Istanbul     644.1
London       613.4
Salalah      153.8
Amman        130.3
Khartoum      77.3
Riyadh        73.9
Cairo          8.8
Name: precipitation, dtype: float64

AVERAGE HUMIDITY BY CITY
city
Zurich      81.24
Munich      75.37
London      73.56
Istanbul    69.52
Salalah     65.02


In [58]:
# ============================================================
# EDA - PART 2
# Seasonality Analysis
# ============================================================

print("=" * 60)
print("EDA - SEASONAL ANALYSIS")
print("=" * 60)

# Create month information
df_2025["month"] = df_2025["date"].dt.month
df_2025["month_name"] = df_2025["date"].dt.month_name()

# Monthly temperature
monthly_temperature = (
    df_2025
    .groupby("month")
    .agg(
        avg_temp=("temp_avg", "mean"),
        max_temp=("temp_max", "max"),
        min_temp=("temp_min", "min")
    )
)

print("\n" + "=" * 60)
print("MONTHLY TEMPERATURE")
print("=" * 60)

print(monthly_temperature.round(2))


# Monthly precipitation
monthly_rain = (
    df_2025
    .groupby("month")
    .agg(
        total_precipitation=("precipitation", "sum"),
        rainy_days=("precipitation", lambda x: (x > 0).sum())
    )
)

print("\n" + "=" * 60)
print("MONTHLY PRECIPITATION")
print("=" * 60)

print(monthly_rain.round(2))


# Monthly humidity
monthly_humidity = (
    df_2025
    .groupby("month")["humidity"]
    .mean()
)

print("\n" + "=" * 60)
print("MONTHLY HUMIDITY")
print("=" * 60)

print(monthly_humidity.round(2))


# Monthly wind
monthly_wind = (
    df_2025
    .groupby("month")["wind_speed"]
    .mean()
)

print("\n" + "=" * 60)
print("MONTHLY WIND SPEED")
print("=" * 60)

print(monthly_wind.round(2))


# Seasonal summary
df_2025["season"] = df_2025["month"].map({
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Summer",
    10: "Autumn",
    11: "Autumn"
})

seasonal_summary = (
    df_2025
    .groupby("season")
    .agg(
        avg_temperature=("temp_avg", "mean"),
        max_temperature=("temp_max", "max"),
        min_temperature=("temp_min", "min"),
        precipitation=("precipitation", "sum"),
        humidity=("humidity", "mean"),
        wind_speed=("wind_speed", "mean")
    )
)

print("\n" + "=" * 60)
print("SEASONAL SUMMARY")
print("=" * 60)

print(seasonal_summary.round(2))

print("\n" + "=" * 60)
print("EDA PART 2 COMPLETED")
print("=" * 60)

EDA - SEASONAL ANALYSIS

MONTHLY TEMPERATURE
       avg_temp  max_temp  min_temp
month                              
1         11.52      36.0      -7.9
2         11.47      37.5      -6.3
3         16.03      43.1      -3.3
4         19.88      44.3      -1.9
5         23.20      44.5       4.0
6         26.82      45.5       9.0
7         26.93      46.2       9.8
8         26.30      45.2       8.3
9         23.79      43.0       7.2
10        20.37      40.6       2.0
11        16.95      38.8      -7.5
12        12.71      37.7      -8.1

MONTHLY PRECIPITATION
       total_precipitation  rainy_days
month                                 
1                    358.7          77
2                    260.0          82
3                    222.5          85
4                    150.3          48
5                    323.3          84
6                    237.4          61
7                    463.4         103
8                    380.8         100
9                    398.2         100

In [59]:
# ============================================================
# EDA - PART 2B
# Compact Seasonal Summary
# ============================================================

print("=" * 60)
print("SEASONAL COMPACT SUMMARY")
print("=" * 60)

print("\nMONTHLY TEMPERATURE")
print(
    monthly_temperature.round(2).to_string()
)

print("\n" + "=" * 60)
print("MONTHLY PRECIPITATION")
print("=" * 60)

print(
    monthly_rain.round(2).to_string()
)

print("\n" + "=" * 60)
print("MONTHLY HUMIDITY")
print("=" * 60)

print(
    monthly_humidity.round(2).to_string()
)

print("\n" + "=" * 60)
print("MONTHLY WIND")
print("=" * 60)

print(
    monthly_wind.round(2).to_string()
)

print("\n" + "=" * 60)
print("SEASONAL SUMMARY")
print("=" * 60)

print(
    seasonal_summary.round(2).to_string()
)

print("\n" + "=" * 60)
print("EDA PART 2B COMPLETED")
print("=" * 60)

SEASONAL COMPACT SUMMARY

MONTHLY TEMPERATURE
       avg_temp  max_temp  min_temp
month                              
1         11.52      36.0      -7.9
2         11.47      37.5      -6.3
3         16.03      43.1      -3.3
4         19.88      44.3      -1.9
5         23.20      44.5       4.0
6         26.82      45.5       9.0
7         26.93      46.2       9.8
8         26.30      45.2       8.3
9         23.79      43.0       7.2
10        20.37      40.6       2.0
11        16.95      38.8      -7.5
12        12.71      37.7      -8.1

MONTHLY PRECIPITATION
       total_precipitation  rainy_days
month                                 
1                    358.7          77
2                    260.0          82
3                    222.5          85
4                    150.3          48
5                    323.3          84
6                    237.4          61
7                    463.4         103
8                    380.8         100
9                    398.2         10

In [60]:
# ============================================================
# EDA - PART 3
# Outlier & Data Quality Analysis
# ============================================================

print("=" * 60)
print("OUTLIER & DATA QUALITY ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# 1. IQR OUTLIER DETECTION
# ------------------------------------------------------------

numeric_columns = [
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

print("\n" + "=" * 60)
print("IQR OUTLIERS")
print("=" * 60)

for column in numeric_columns:

    Q1 = df_2025[column].quantile(0.25)
    Q3 = df_2025[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_2025[
        (df_2025[column] < lower_bound) |
        (df_2025[column] > upper_bound)
    ]

    print(f"\n{column}")
    print("Q1:", round(Q1, 2))
    print("Q3:", round(Q3, 2))
    print("IQR:", round(IQR, 2))
    print("Lower bound:", round(lower_bound, 2))
    print("Upper bound:", round(upper_bound, 2))
    print("Outlier count:", len(outliers))


# ------------------------------------------------------------
# 2. EXTREME TEMPERATURE DAYS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXTREME TEMPERATURE DAYS")
print("=" * 60)

hot_days = df_2025[
    df_2025["temp_max"] >= 45
][
    [
        "date",
        "city",
        "temp_avg",
        "temp_max",
        "temp_min"
    ]
].sort_values("temp_max", ascending=False)

print("\nDays with maximum temperature >= 45°C:")
print(hot_days.to_string(index=False))


cold_days = df_2025[
    df_2025["temp_min"] <= -5
][
    [
        "date",
        "city",
        "temp_avg",
        "temp_max",
        "temp_min"
    ]
].sort_values("temp_min")

print("\nDays with minimum temperature <= -5°C:")
print(cold_days.to_string(index=False))


# ------------------------------------------------------------
# 3. EXTREME PRECIPITATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXTREME PRECIPITATION")
print("=" * 60)

heavy_rain = df_2025[
    df_2025["precipitation"] >= 20
][
    [
        "date",
        "city",
        "precipitation"
    ]
].sort_values(
    "precipitation",
    ascending=False
)

print(
    heavy_rain.to_string(index=False)
)


# ------------------------------------------------------------
# 4. EXTREME WIND
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXTREME WIND")
print("=" * 60)

strong_wind = df_2025[
    df_2025["wind_speed"] >= 25
][
    [
        "date",
        "city",
        "wind_speed"
    ]
].sort_values(
    "wind_speed",
    ascending=False
)

print(
    strong_wind.to_string(index=False)
)


# ------------------------------------------------------------
# 5. BASIC DATA QUALITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print("Missing values:")
print(df_2025.isna().sum())

print("\nDuplicate rows:")
print(df_2025.duplicated().sum())

print("\nInvalid temperature relationships:")

invalid_temperature = df_2025[
    (df_2025["temp_min"] > df_2025["temp_avg"]) |
    (df_2025["temp_avg"] > df_2025["temp_max"])
]

print(len(invalid_temperature))

print("\nInvalid humidity:")
print(
    len(
        df_2025[
            (df_2025["humidity"] < 0) |
            (df_2025["humidity"] > 100)
        ]
    )
)

print("\nInvalid precipitation:")
print(
    len(
        df_2025[
            df_2025["precipitation"] < 0
        ]
    )
)

print("\nInvalid wind speed:")
print(
    len(
        df_2025[
            df_2025["wind_speed"] < 0
        ]
    )
)

print("\n" + "=" * 60)
print("EDA PART 3 COMPLETED")
print("=" * 60)

OUTLIER & DATA QUALITY ANALYSIS

IQR OUTLIERS

temp_avg
Q1: 11.9
Q3: 27.2
IQR: 15.3
Lower bound: -11.05
Upper bound: 50.15
Outlier count: 0

temp_max
Q1: 16.1
Q3: 32.8
IQR: 16.7
Lower bound: -8.95
Upper bound: 57.85
Outlier count: 0

temp_min
Q1: 8.5
Q3: 22.7
IQR: 14.2
Lower bound: -12.8
Upper bound: 44.0
Outlier count: 0

precipitation
Q1: 0.0
Q3: 0.2
IQR: 0.2
Lower bound: -0.3
Upper bound: 0.5
Outlier count: 687

humidity
Q1: 37.0
Q3: 78.0
IQR: 41.0
Lower bound: -24.5
Upper bound: 139.5
Outlier count: 0

wind_speed
Q1: 6.1
Q3: 10.6
IQR: 4.5
Lower bound: -0.65
Upper bound: 17.35
Outlier count: 122

EXTREME TEMPERATURE DAYS

Days with maximum temperature >= 45°C:
      date     city  temp_avg  temp_max  temp_min
2025-07-30   Riyadh      39.9      46.2      33.3
2025-07-29   Riyadh      40.1      45.6      33.3
2025-07-28   Riyadh      40.1      45.6      33.5
2025-06-16   Riyadh      39.3      45.5      32.5
2025-06-13 Khartoum      38.6      45.4      31.0
2025-08-11   Riyadh      40.

In [61]:
# ============================================================
# EDA - PART 4
# Correlation & Relationships
# ============================================================

print("=" * 60)
print("CORRELATION & RELATIONSHIP ANALYSIS")
print("=" * 60)

numeric_columns = [
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

correlation = df_2025[numeric_columns].corr()

print("\nCorrelation Matrix:")
print(correlation.round(2).to_string())

print("\n" + "=" * 60)
print("STRONGEST CORRELATIONS")
print("=" * 60)

# Keep only unique pairs
pairs = []

for i in range(len(numeric_columns)):
    for j in range(i + 1, len(numeric_columns)):

        col1 = numeric_columns[i]
        col2 = numeric_columns[j]

        value = correlation.loc[col1, col2]

        pairs.append(
            {
                "variable_1": col1,
                "variable_2": col2,
                "correlation": value
            }
        )

correlation_pairs = pd.DataFrame(pairs)

correlation_pairs["abs_correlation"] = (
    correlation_pairs["correlation"].abs()
)

correlation_pairs = (
    correlation_pairs
    .sort_values(
        "abs_correlation",
        ascending=False
    )
    .drop(columns="abs_correlation")
)

print(
    correlation_pairs
    .round(2)
    .to_string(index=False)
)

print("\n" + "=" * 60)
print("CITY TEMPERATURE RANGE")
print("=" * 60)

temperature_range = (
    df_2025
    .groupby("city")
    .agg(
        avg_temp=("temp_avg", "mean"),
        avg_max=("temp_max", "mean"),
        avg_min=("temp_min", "mean")
    )
)

temperature_range["daily_temperature_range"] = (
    temperature_range["avg_max"]
    - temperature_range["avg_min"]
)

print(
    temperature_range
    .sort_values(
        "daily_temperature_range",
        ascending=False
    )
    .round(2)
    .to_string()
)

print("\n" + "=" * 60)
print("EDA PART 4 COMPLETED")
print("=" * 60)

CORRELATION & RELATIONSHIP ANALYSIS

Correlation Matrix:
               temp_avg  temp_max  temp_min  precipitation  humidity  wind_speed
temp_avg           1.00      0.99      0.98          -0.22     -0.73        0.01
temp_max           0.99      1.00      0.95          -0.25     -0.76       -0.03
temp_min           0.98      0.95      1.00          -0.17     -0.65        0.05
precipitation     -0.22     -0.25     -0.17           1.00      0.34        0.14
humidity          -0.73     -0.76     -0.65           0.34      1.00       -0.04
wind_speed         0.01     -0.03      0.05           0.14     -0.04        1.00

STRONGEST CORRELATIONS
   variable_1    variable_2  correlation
     temp_avg      temp_max         0.99
     temp_avg      temp_min         0.98
     temp_max      temp_min         0.95
     temp_max      humidity        -0.76
     temp_avg      humidity        -0.73
     temp_min      humidity        -0.65
precipitation      humidity         0.34
     temp_max precipitat

In [67]:
from pathlib import Path
import sys

# Make the project root importable from a notebook session
for candidate in [Path.cwd(), Path.cwd().parent]:
    candidate = candidate.resolve()
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

try:
    from src.eda import run_eda
except ModuleNotFoundError:
    # Fallback for notebook environments launched from a subfolder
    repo_root = next(
        (p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists()),
        None
    )
    if repo_root is not None and str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root.resolve()))
    from src.eda import run_eda

eda_results = run_eda(
    df_2025,
    year=2025
)

print(eda_results["overview"])

{'rows': 3285, 'columns': 11, 'cities': 9, 'start_date': Timestamp('2025-01-01 00:00:00'), 'end_date': Timestamp('2025-12-31 00:00:00'), 'memory_bytes': np.int64(740838)}


In [68]:
print(df_2025.columns.tolist())

['date', 'city', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'month', 'month_name', 'season']


In [36]:
import importlib
import src.transformation

importlib.reload(src.transformation)

from src.transformation import prepare_dates

df_transformed = prepare_dates(df_2025)

print(df_transformed.dtypes)

date             datetime64[us]
city                        str
temp_avg                float64
temp_max                float64
temp_min                float64
precipitation           float64
humidity                  int64
wind_speed              float64
dtype: object


In [37]:
from src.transformation import rename_columns

raw_test = pd.DataFrame({
    "time": ["2025-01-01"],
    "temperature_2m_mean": [20.5],
    "temperature_2m_max": [25.0],
    "temperature_2m_min": [15.0],
    "precipitation_sum": [1.2],
    "relative_humidity_2m_mean": [60],
    "wind_speed_10m_mean": [8.5],
    "city": ["Cairo"],
})

renamed_test = rename_columns(raw_test)

print(renamed_test.columns.tolist())
print(renamed_test)

['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city']
         date  temp_avg  temp_max  temp_min  precipitation  humidity  \
0  2025-01-01      20.5      25.0      15.0            1.2        60   

   wind_speed   city  
0         8.5  Cairo  


In [29]:
for name in ["cities", "locations_df", "weather_raw"]:
    print(name, "->", "defined" if name in globals() else "MISSING")

cities -> MISSING
locations_df -> MISSING
weather_raw -> MISSING


In [58]:
from pathlib import Path
import sys

repo_root = next(
    (
        p.resolve()
        for p in [Path.cwd(), Path.cwd().parent]
        if p.exists() and (p / "src").exists()
    ),
    None,
)

if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.geocoding import get_locations

locations_df = get_locations()

print(locations_df)
print(locations_df.shape)

       city              country  latitude  longitude         timezone
0    Riyadh         Saudi Arabia  24.68773   46.72185      Asia/Riyadh
1   Salalah                 Oman  17.01505   54.09237      Asia/Muscat
2     Amman               Jordan  31.95522   35.94503       Asia/Amman
3  Istanbul  Republic of Türkiye  41.01384   28.94966  Europe/Istanbul
4    London       United Kingdom  51.50853   -0.12574    Europe/London
5    Munich              Germany  48.13743   11.57549    Europe/Berlin
6    Zurich          Switzerland  47.36667    8.55000    Europe/Zurich
7     Cairo                Egypt  30.06263   31.24967     Africa/Cairo
8  Khartoum                Sudan  15.55177   32.53241  Africa/Khartoum
(9, 5)


In [41]:
import os
print(os.getcwd())
print(os.listdir("src"))

c:\Users\DELL\api_data_pipeline
['eda.py', 'geocoding', 'transformation.py', '__pycache__']


In [42]:
import os
print(os.path.isdir("src/geocoding"))
print(os.path.isfile("src/geocoding"))

False
True


In [43]:
import os
os.rename("src/geocoding", "src/geocoding.py")
print(os.listdir("src"))

['eda.py', 'geocoding.py', 'transformation.py', '__pycache__']


In [45]:
import importlib
import src.geocoding

importlib.reload(src.geocoding)

from src.geocoding import get_locations

locations_df = get_locations()

print(locations_df)
print("\nShape:", locations_df.shape)

       city              country  latitude  longitude         timezone
0    Riyadh         Saudi Arabia  24.68773   46.72185      Asia/Riyadh
1   Salalah                 Oman  17.01505   54.09237      Asia/Muscat
2     Amman               Jordan  31.95522   35.94503       Asia/Amman
3  Istanbul  Republic of Türkiye  41.01384   28.94966  Europe/Istanbul
4    London       United Kingdom  51.50853   -0.12574    Europe/London
5    Munich              Germany  48.13743   11.57549    Europe/Berlin
6    Zurich          Switzerland  47.36667    8.55000    Europe/Zurich
7     Cairo                Egypt  30.06263   31.24967     Africa/Cairo
8  Khartoum                Sudan  15.55177   32.53241  Africa/Khartoum

Shape: (9, 5)


In [46]:
from src.transformation import build_dim_city

dim_city = build_dim_city(locations_df)

print(dim_city)
print("\nShape:", dim_city.shape)

   city_key      city              country  latitude  longitude
0         1    Riyadh         Saudi Arabia  24.68773   46.72185
1         2   Salalah                 Oman  17.01505   54.09237
2         3     Amman               Jordan  31.95522   35.94503
3         4  Istanbul  Republic of Türkiye  41.01384   28.94966
4         5    London       United Kingdom  51.50853   -0.12574
5         6    Munich              Germany  48.13743   11.57549
6         7    Zurich          Switzerland  47.36667    8.55000
7         8     Cairo                Egypt  30.06263   31.24967
8         9  Khartoum                Sudan  15.55177   32.53241

Shape: (9, 5)


In [62]:
import importlib
import src.transformation

importlib.reload(src.transformation)

from src.transformation import attach_city_key

weather_with_city_key = attach_city_key(
    df_transformed,
    dim_city
)

print(weather_with_city_key.head())
print("\nColumns:")
print(weather_with_city_key.columns.tolist())

print("\nShape:")
print(weather_with_city_key.shape)

        date  temp_avg  temp_max  temp_min  precipitation  humidity  \
0 2025-01-01       8.1      12.6       5.1            0.0        78   
1 2025-01-01      14.7      20.0       9.9            0.0        64   
2 2025-01-01       5.9       9.0       3.0            0.0        90   
3 2025-01-01      21.9      28.7      15.8            0.0        27   
4 2025-01-01       9.0      11.8       5.3           10.5        87   

   wind_speed  city_key  
0         2.9         3  
1         5.2         8  
2         3.3         4  
3        14.4         9  
4        21.8         5  

Columns:
['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city_key']

Shape:
(3285, 8)


In [64]:
import pandas as pd

dim_date = (
    df_2025[["date"]]
    .copy()
)

dim_date["date"] = pd.to_datetime(dim_date["date"])

dim_date = (
    dim_date
    .drop_duplicates()
    .sort_values("date")
)

dim_date["date_key"] = (
    dim_date["date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()

dim_date = dim_date[
    [
        "date_key",
        "date",
        "year",
        "month",
        "month_name"
    ]
]

print(dim_date.head())
print("\nShape:", dim_date.shape)

    date_key       date  year  month month_name
0   20250101 2025-01-01  2025      1    January
9   20250102 2025-01-02  2025      1    January
18  20250103 2025-01-03  2025      1    January
27  20250104 2025-01-04  2025      1    January
36  20250105 2025-01-05  2025      1    January

Shape: (365, 5)


In [65]:
import importlib
import src.transformation

importlib.reload(src.transformation)

from src.transformation import build_fact_weather

fact_weather = build_fact_weather(
    df_2025,
    dim_city,
    dim_date
)

print(fact_weather.head())
print("\nColumns:")
print(fact_weather.columns.tolist())
print("\nShape:")
print(fact_weather.shape)

   temp_avg  temp_max  temp_min  precipitation  humidity  wind_speed  \
0       8.1      12.6       5.1            0.0        78         2.9   
1      14.7      20.0       9.9            0.0        64         5.2   
2       5.9       9.0       3.0            0.0        90         3.3   
3      21.9      28.7      15.8            0.0        27        14.4   
4       9.0      11.8       5.3           10.5        87        21.8   

   city_key  date_key  
0         3  20250101  
1         8  20250101  
2         4  20250101  
3         9  20250101  
4         5  20250101  

Columns:
['temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city_key', 'date_key']

Shape:
(3285, 8)


In [66]:
# Validate fact table grain

print("Rows:", len(fact_weather))

print(
    "Unique city-date combinations:",
    fact_weather[["city_key", "date_key"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate city-date rows:",
    fact_weather.duplicated(
        subset=["city_key", "date_key"]
    ).sum()
)

print(
    "Missing values:",
    fact_weather.isna().sum().sum()
)

Rows: 3285
Unique city-date combinations: 3285
Duplicate city-date rows: 0
Missing values: 0


In [67]:
fact_weather = (
    fact_weather
    .sort_values(["date_key", "city_key"])
    .reset_index(drop=True)
)

fact_weather.insert(
    0,
    "weather_id",
    range(1, len(fact_weather) + 1)
)

print(fact_weather.head())

print("\nColumns:")
print(fact_weather.columns.tolist())

print("\nShape:")
print(fact_weather.shape)

   weather_id  temp_avg  temp_max  temp_min  precipitation  humidity  \
0           1      13.8      17.6       9.7            0.0        34   
1           2      21.9      26.2      17.7            0.0        68   
2           3       8.1      12.6       5.1            0.0        78   
3           4       5.9       9.0       3.0            0.0        90   
4           5       9.0      11.8       5.3           10.5        87   

   wind_speed  city_key  date_key  
0         8.6         1  20250101  
1         6.1         2  20250101  
2         2.9         3  20250101  
3         3.3         4  20250101  
4        21.8         5  20250101  

Columns:
['weather_id', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city_key', 'date_key']

Shape:
(3285, 9)


In [ ]:
import dotenv
import requests

print("Environment is ready!")

Environment is ready!


In [ ]:
import requests
import sys
!{sys.executable} -m pip install pandas
import pandas as pd

print("Requests:", requests.__version__)
print("Pandas:", pd.__version__)

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.8 MB 3.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.8 MB 4.2 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.8 MB 4.3 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.8 MB 4.4 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.8 MB 4.5 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.8 MB 4.5 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/9.8 MB 4.6 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.8 MB 4.6 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 4.5 MB/s  0:00:02
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -- -----------------------------

In [ ]:
weather_url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.06263,
    "longitude": 31.24967,
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "relative_humidity_2m_mean",
        "wind_speed_10m_mean"
    ],
    "timezone": "auto",
    "past_days": 7
}

response = requests.get(
    weather_url,
    params=params,
    timeout=10
)

print("Status:", response.status_code)

Status: 200


In [ ]:
data = response.json()

print(data.keys())
print(data["daily"].keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
dict_keys(['time', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'relative_humidity_2m_mean', 'wind_speed_10m_mean'])


In [ ]:
from pathlib import Path
import json

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

raw_file = raw_dir / "cairo_weather_raw.json"

with open(raw_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved: {raw_file}")

Saved: data\raw\cairo_weather_raw.json


In [ ]:
cities = [
    "Riyadh",
    "Salalah",
    "Amman",
    "Istanbul",
    "London",
    "Munich",
    "Zurich",
    "Cairo",
    "Khartoum"
]

print(cities)
print("Number of cities:", len(cities))

['Riyadh', 'Salalah', 'Amman', 'Istanbul', 'London', 'Munich', 'Zurich', 'Cairo', 'Khartoum']
Number of cities: 9


In [ ]:
import requests
import pandas as pd

geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

locations = []

for city in cities:
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        geocoding_url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    result = response.json()

    if "results" not in result or not result["results"]:
        print(f"Not found: {city}")
        continue

    location = result["results"][0]

    locations.append({
        "city": city,
        "country": location["country"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "timezone": location["timezone"]
    })

locations_df = pd.DataFrame(locations)

print(locations_df)

       city              country  latitude  longitude         timezone
0    Riyadh         Saudi Arabia  24.68773   46.72185      Asia/Riyadh
1   Salalah                 Oman  17.01505   54.09237      Asia/Muscat
2     Amman               Jordan  31.95522   35.94503       Asia/Amman
3  Istanbul  Republic of Türkiye  41.01384   28.94966  Europe/Istanbul
4    London       United Kingdom  51.50853   -0.12574    Europe/London
5    Munich              Germany  48.13743   11.57549    Europe/Berlin
6    Zurich          Switzerland  47.36667    8.55000    Europe/Zurich
7     Cairo                Egypt  30.06263   31.24967     Africa/Cairo
8  Khartoum                Sudan  15.55177   32.53241  Africa/Khartoum


In [ ]:

weather_raw = {}

for _, location in locations_df.iterrows():

    city = location["city"]
    latitude = location["latitude"]
    longitude = location["longitude"]

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "wind_speed_10m_mean"
        ],
        "timezone": "auto",
        "past_days": 7
    }

    response = requests.get(
        weather_url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    weather_raw[city] = response.json()

    print(f"{city}: {response.status_code}")

Riyadh: 200
Salalah: 200
Amman: 200
Istanbul: 200
London: 200
Munich: 200
Zurich: 200
Cairo: 200
Khartoum: 200


In [ ]:
from pathlib import Path
import json

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

for city, weather_data in weather_raw.items():
    file_path = raw_dir / f"{city.lower()}_weather_raw.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(weather_data, f, ensure_ascii=False, indent=2)

    print(f"Saved: {file_path}")

Saved: data\raw\riyadh_weather_raw.json
Saved: data\raw\salalah_weather_raw.json
Saved: data\raw\amman_weather_raw.json
Saved: data\raw\istanbul_weather_raw.json
Saved: data\raw\london_weather_raw.json
Saved: data\raw\munich_weather_raw.json
Saved: data\raw\zurich_weather_raw.json
Saved: data\raw\cairo_weather_raw.json
Saved: data\raw\khartoum_weather_raw.json


In [ ]:
all_data = []

for city, weather_data in weather_raw.items():

    daily = weather_data["daily"]

    city_df = pd.DataFrame(daily)

    city_df["city"] = city

    all_data.append(city_df)

df = pd.concat(all_data, ignore_index=True)

print(df.shape)
print(df.head())

(126, 8)
         time  temperature_2m_mean  temperature_2m_max  temperature_2m_min  \
0  2026-09-06                 35.3                39.5                29.6   
1  2026-09-07                 37.0                42.1                31.3   
2  2026-09-08                 37.9                43.1                32.4   
3  2026-09-09                 38.6                43.2                33.9   
4  2026-09-10                 38.6                42.7                33.4   

   precipitation_sum  relative_humidity_2m_mean  wind_speed_10m_mean    city  
0                0.0                         21                  5.5  Riyadh  
1                0.0                         14                  9.6  Riyadh  
2                0.0                         14                  8.6  Riyadh  
3                0.0                         13                  6.4  Riyadh  
4                0.0                         13                  6.1  Riyadh  


In [ ]:
print(df.groupby("city")["time"].count())

city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: time, dtype: int64


In [ ]:
df["time"] = pd.to_datetime(df["time"])

print(df.dtypes)

time                         datetime64[us]
temperature_2m_mean                 float64
temperature_2m_max                  float64
temperature_2m_min                  float64
precipitation_sum                   float64
relative_humidity_2m_mean             int64
wind_speed_10m_mean                 float64
city                                    str
dtype: object


In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicates:", df.duplicated().sum())

Shape: (126, 8)

Missing values:
time                         0
temperature_2m_mean          0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
relative_humidity_2m_mean    0
wind_speed_10m_mean          0
city                         0
dtype: int64

Duplicates: 0


In [ ]:
print("Temperature:")
print("Min:", df["temperature_2m_min"].min())
print("Max:", df["temperature_2m_max"].max())

print("\nHumidity:")
print("Min:", df["relative_humidity_2m_mean"].min())
print("Max:", df["relative_humidity_2m_mean"].max())

print("\nPrecipitation:")
print("Min:", df["precipitation_sum"].min())

print("\nWind Speed:")
print("Min:", df["wind_speed_10m_mean"].min())

Temperature:
Min: 9.2
Max: 44.6

Humidity:
Min: 8
Max: 92

Precipitation:
Min: 0.0

Wind Speed:
Min: 2.7


In [ ]:
print("Temperature:")
print("Min:", df["temperature_2m_min"].min())
print("Max:", df["temperature_2m_max"].max())

print("\nHumidity:")
print("Min:", df["relative_humidity_2m_mean"].min())
print("Max:", df["relative_humidity_2m_mean"].max())

print("\nPrecipitation:")
print("Min:", df["precipitation_sum"].min())

print("\nWind Speed:")
print("Min:", df["wind_speed_10m_mean"].min())

Temperature:
Min: 9.2
Max: 44.6

Humidity:
Min: 8
Max: 92

Precipitation:
Min: 0.0

Wind Speed:
Min: 2.7


In [ ]:
print(df.groupby("city")["time"].count())

city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: time, dtype: int64


In [ ]:
print("City-Date duplicates:",
      df.duplicated(subset=["city", "time"]).sum())

City-Date duplicates: 0


In [ ]:
df = df.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temp_avg",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "precipitation",
    "relative_humidity_2m_mean": "humidity",
    "wind_speed_10m_mean": "wind_speed"
})

print(df.columns.tolist())

['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city']


In [ ]:
print(df.columns.tolist())

['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city']


In [ ]:
dim_city = locations_df[
    ["city", "country", "latitude", "longitude"]
].copy()

dim_city.insert(0, "city_key", range(1, len(dim_city) + 1))

print(dim_city)

   city_key      city              country  latitude  longitude
0         1    Riyadh         Saudi Arabia  24.68773   46.72185
1         2   Salalah                 Oman  17.01505   54.09237
2         3     Amman               Jordan  31.95522   35.94503
3         4  Istanbul  Republic of Türkiye  41.01384   28.94966
4         5    London       United Kingdom  51.50853   -0.12574
5         6    Munich              Germany  48.13743   11.57549
6         7    Zurich          Switzerland  47.36667    8.55000
7         8     Cairo                Egypt  30.06263   31.24967
8         9  Khartoum                Sudan  15.55177   32.53241


In [ ]:
dim_date = df[["date"]].drop_duplicates().sort_values("date").copy()

dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()

print(dim_date)

         date  date_key  year  month month_name
56 2026-09-05  20260905  2026      9  September
0  2026-09-06  20260906  2026      9  September
1  2026-09-07  20260907  2026      9  September
2  2026-09-08  20260908  2026      9  September
3  2026-09-09  20260909  2026      9  September
4  2026-09-10  20260910  2026      9  September
5  2026-09-11  20260911  2026      9  September
6  2026-09-12  20260912  2026      9  September
7  2026-09-13  20260913  2026      9  September
8  2026-09-14  20260914  2026      9  September
9  2026-09-15  20260915  2026      9  September
10 2026-09-16  20260916  2026      9  September
11 2026-09-17  20260917  2026      9  September
12 2026-09-18  20260918  2026      9  September
13 2026-09-19  20260919  2026      9  September


In [ ]:
print("DataFrame rows:", len(df))
print("Unique dates:", df["date"].nunique())
print("Expected rows:", len(dim_city) * dim_date["date"].nunique())

print("\nRows per city:")
print(df.groupby("city")["date"].count())

DataFrame rows: 126
Unique dates: 15
Expected rows: 135

Rows per city:
city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: date, dtype: int64


In [ ]:
print("DataFrame rows:", len(df))
print("Unique dates:", df["date"].nunique())
print("Expected rows:", len(dim_city) * dim_date["date"].nunique())

print("\nRows per city:")

print(df.groupby("city")["date"].count())

DataFrame rows: 126
Unique dates: 15
Expected rows: 135

Rows per city:
city
Amman       14
Cairo       14
Istanbul    14
Khartoum    14
London      14
Munich      14
Riyadh      14
Salalah     14
Zurich      14
Name: date, dtype: int64


In [ ]:
expected_dates = set(df["date"].unique())

for city in df["city"].unique():
    city_dates = set(df.loc[df["city"] == city, "date"])
    missing_dates = sorted(expected_dates - city_dates)

    if missing_dates:
        print(city, "missing:", missing_dates)

Riyadh missing: [Timestamp('2026-09-05 00:00:00')]
Salalah missing: [Timestamp('2026-09-05 00:00:00')]
Amman missing: [Timestamp('2026-09-05 00:00:00')]
Istanbul missing: [Timestamp('2026-09-05 00:00:00')]
London missing: [Timestamp('2026-09-19 00:00:00')]
Munich missing: [Timestamp('2026-09-05 00:00:00')]
Zurich missing: [Timestamp('2026-09-05 00:00:00')]
Cairo missing: [Timestamp('2026-09-05 00:00:00')]
Khartoum missing: [Timestamp('2026-09-05 00:00:00')]


In [ ]:
from pathlib import Path
import json

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

for city, weather_data in weather_raw.items():
    file_path = raw_dir / f"{city.lower()}_weather_raw.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(weather_data, f, ensure_ascii=False, indent=2)

    print(f"Saved: {file_path}")

Saved: data\raw\riyadh_weather_raw.json
Saved: data\raw\salalah_weather_raw.json
Saved: data\raw\amman_weather_raw.json
Saved: data\raw\istanbul_weather_raw.json
Saved: data\raw\london_weather_raw.json
Saved: data\raw\munich_weather_raw.json
Saved: data\raw\zurich_weather_raw.json
Saved: data\raw\cairo_weather_raw.json
Saved: data\raw\khartoum_weather_raw.json


In [ ]:
from pathlib import Path

batch_dir = Path("data/batches/2025-01")

raw_batch_dir = batch_dir / "raw"
checkpoint_dir = batch_dir / "checkpoint"

raw_batch_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Batch directory:", batch_dir)
print("Raw directory:", raw_batch_dir)
print("Checkpoint directory:", checkpoint_dir)

Batch directory: data\batches\2025-01
Raw directory: data\batches\2025-01\raw
Checkpoint directory: data\batches\2025-01\checkpoint


In [ ]:
import json
import requests
import pandas as pd
from pathlib import Path
from calendar import monthrange

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

YEAR = 2025

HISTORICAL_WEATHER_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)

DAILY_VARIABLES = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "relative_humidity_2m_mean",
    "wind_speed_10m_mean"
]

BASE_BATCH_DIR = Path("data/batches")


# ============================================================
# 2. FUNCTION: EXTRACT ONE MONTH
# ============================================================

def extract_month(year, month, locations_df):

    month_str = f"{year}-{month:02d}"

    start_date = f"{year}-{month:02d}-01"

    last_day = monthrange(year, month)[1]

    end_date = f"{year}-{month:02d}-{last_day:02d}"

    print("\n")
    print("=" * 60)
    print(f"PROCESSING BATCH: {month_str}")
    print(f"Date range: {start_date} → {end_date}")
    print("=" * 60)

    # --------------------------------------------------------
    # Batch directories
    # --------------------------------------------------------

    batch_dir = BASE_BATCH_DIR / month_str

    raw_dir = batch_dir / "raw"

    checkpoint_dir = batch_dir / "checkpoint"

    raw_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_file = (
        checkpoint_dir / f"weather_{month_str}.csv"
    )

    # --------------------------------------------------------
    # Resume protection
    # --------------------------------------------------------

    if checkpoint_file.exists():

        print(
            f"CHECKPOINT ALREADY EXISTS: "
            f"{checkpoint_file}"
        )

        existing_df = pd.read_csv(
            checkpoint_file
        )

        print(
            f"Skipping API extraction."
            f" Existing rows: {len(existing_df)}"
        )

        return existing_df

    # --------------------------------------------------------
    # Extract each city
    # --------------------------------------------------------

    all_data = []

    for _, location in locations_df.iterrows():

        city = location["city"]

        latitude = location["latitude"]

        longitude = location["longitude"]

        raw_file = (
            raw_dir /
            f"{city.lower()}_weather_raw.json"
        )

        print(f"\nRequesting: {city}")

        params = {
            "latitude": latitude,
            "longitude": longitude,
            "daily": DAILY_VARIABLES,
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date
        }

        try:

            response = requests.get(
                HISTORICAL_WEATHER_URL,
                params=params,
                timeout=30
            )

            print(
                f"Status: {response.status_code}"
            )

            response.raise_for_status()

            weather_data = response.json()

            # ------------------------------------------------
            # Save RAW immediately
            # ------------------------------------------------

            with open(
                raw_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    weather_data,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            # ------------------------------------------------
            # Convert RAW → DataFrame
            # ------------------------------------------------

            daily = weather_data["daily"]

            city_df = pd.DataFrame(daily)

            city_df["city"] = city

            all_data.append(city_df)

            print(
                f"Saved: {raw_file} | "
                f"Days: {len(city_df)}"
            )

        except Exception as e:

            print(
                f"ERROR in {city}: {e}"
            )

            print(
                "Stopping this batch."
            )

            raise

    # ========================================================
    # 3. COMBINE CITIES
    # ========================================================

    batch_df = pd.concat(
        all_data,
        ignore_index=True
    )

    # ========================================================
    # 4. RENAME COLUMNS
    # ========================================================

    batch_df = batch_df.rename(
        columns={
            "time": "date",
            "temperature_2m_mean": "temp_avg",
            "temperature_2m_max": "temp_max",
            "temperature_2m_min": "temp_min",
            "precipitation_sum": "precipitation",
            "relative_humidity_2m_mean": "humidity",
            "wind_speed_10m_mean": "wind_speed"
        }
    )

    # ========================================================
    # 5. DATE TYPE
    # ========================================================

    batch_df["date"] = pd.to_datetime(
        batch_df["date"]
    )

    # ========================================================
    # 6. COLUMN ORDER
    # ========================================================

    batch_df = batch_df[
        [
            "date",
            "city",
            "temp_avg",
            "temp_max",
            "temp_min",
            "precipitation",
            "humidity",
            "wind_speed"
        ]
    ]

    # ========================================================
    # 7. SAVE CSV CHECKPOINT
    # ========================================================

    batch_df.to_csv(
        checkpoint_file,
        index=False
    )

    # ========================================================
    # 8. VERIFY BATCH
    # ========================================================

    expected_rows = (
        len(locations_df) *
        last_day
    )

    actual_rows = len(batch_df)

    print("\n")
    print("-" * 60)
    print(f"BATCH COMPLETED: {month_str}")
    print("-" * 60)

    print(
        f"Expected rows: {expected_rows}"
    )

    print(
        f"Actual rows:   {actual_rows}"
    )

    print(
        f"Cities:        "
        f"{batch_df['city'].nunique()}"
    )

    print(
        f"Unique dates:  "
        f"{batch_df['date'].nunique()}"
    )

    print(
        f"CSV checkpoint: "
        f"{checkpoint_file}"
    )

    # --------------------------------------------------------
    # City row verification
    # --------------------------------------------------------

    print("\nRows per city:")

    print(
        batch_df
        .groupby("city")["date"]
        .count()
    )

    # --------------------------------------------------------
    # Basic validation
    # --------------------------------------------------------

    if actual_rows != expected_rows:

        raise ValueError(
            f"Row count mismatch "
            f"for {month_str}: "
            f"expected {expected_rows}, "
            f"got {actual_rows}"
        )

    print(
        "\nSTATUS: SUCCESS"
    )

    return batch_df


# ============================================================
# 9. RUN ALL 12 MONTHS
# ============================================================

monthly_results = {}

for month in range(1, 13):

    month_df = extract_month(
        YEAR,
        month,
        locations_df
    )

    monthly_results[
        f"{YEAR}-{month:02d}"
    ] = month_df


# ============================================================
# 10. FINAL YEAR SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("2025 FULL YEAR EXTRACTION COMPLETED")
print("=" * 60)

total_rows = 0

for month_str, month_df in monthly_results.items():

    rows = len(month_df)

    total_rows += rows

    print(
        f"{month_str}: {rows} rows"
    )

print("-" * 60)

print(
    f"TOTAL ROWS: {total_rows}"
)

expected_year_rows = (
    len(locations_df) * 365
)

print(
    f"EXPECTED:   {expected_year_rows}"
)

print("=" * 60)

if total_rows == expected_year_rows:

    print(
        "YEAR STATUS: SUCCESS"
    )

else:

    print(
        "YEAR STATUS: CHECK REQUIRED"
    )



PROCESSING BATCH: 2025-01
Date range: 2025-01-01 → 2025-01-31
CHECKPOINT ALREADY EXISTS: data\batches\2025-01\checkpoint\weather_2025-01.csv
Skipping API extraction. Existing rows: 279


PROCESSING BATCH: 2025-02
Date range: 2025-02-01 → 2025-02-28

Requesting: Riyadh
Status: 200
Saved: data\batches\2025-02\raw\riyadh_weather_raw.json | Days: 28

Requesting: Salalah
Status: 200
Saved: data\batches\2025-02\raw\salalah_weather_raw.json | Days: 28

Requesting: Amman
Status: 200
Saved: data\batches\2025-02\raw\amman_weather_raw.json | Days: 28

Requesting: Istanbul
Status: 200
Saved: data\batches\2025-02\raw\istanbul_weather_raw.json | Days: 28

Requesting: London
Status: 200
Saved: data\batches\2025-02\raw\london_weather_raw.json | Days: 28

Requesting: Munich
Status: 200
Saved: data\batches\2025-02\raw\munich_weather_raw.json | Days: 28

Requesting: Zurich
Status: 200
Saved: data\batches\2025-02\raw\zurich_weather_raw.json | Days: 28

Requesting: Cairo
Status: 200
Saved: data\batches\2

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

YEAR = 2025

BASE_BATCH_DIR = Path("data/batches")

FINAL_DATA_DIR = Path("data/final")
FINAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

final_csv = FINAL_DATA_DIR / "weather_2025.csv"


# ============================================================
# 2. CANONICAL CITY NAMES
# ============================================================

city_name_map = {
    "riyadh": "Riyadh",
    "salalah": "Salalah",
    "amman": "Amman",
    "istanbul": "Istanbul",
    "london": "London",
    "munich": "Munich",
    "zurich": "Zurich",
    "cairo": "Cairo",
    "khartoum": "Khartoum"
}

expected_cities = set(
    city_name_map.values()
)


# ============================================================
# 3. LOAD ALL 12 CHECKPOINTS
# ============================================================

monthly_data = []

print("=" * 60)
print("LOADING 2025 MONTHLY CHECKPOINTS")
print("=" * 60)

for month in range(1, 13):

    month_str = f"{YEAR}-{month:02d}"

    checkpoint_file = (
        BASE_BATCH_DIR
        / month_str
        / "checkpoint"
        / f"weather_{month_str}.csv"
    )

    print(f"\nLoading: {month_str}")

    if not checkpoint_file.exists():

        raise FileNotFoundError(
            f"Missing checkpoint: {checkpoint_file}"
        )

    month_df = pd.read_csv(
        checkpoint_file
    )

    month_df["date"] = pd.to_datetime(
        month_df["date"]
    )

    print(
        f"Rows loaded: {len(month_df)}"
    )

    # ========================================================
    # 4. STANDARDIZE CITY NAMES
    # ========================================================

    month_df["city"] = (
        month_df["city"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(city_name_map)
    )

    # Check for unmapped cities
    if month_df["city"].isna().any():

        bad_values = (
            month_df.loc[
                month_df["city"].isna()
            ]
        )

        raise ValueError(
            f"Unknown city names found in "
            f"{month_str}:\n{bad_values}"
        )

    monthly_data.append(
        month_df
    )


# ============================================================
# 5. COMBINE ALL MONTHS
# ============================================================

df_2025 = pd.concat(
    monthly_data,
    ignore_index=True
)

print("\n")
print("=" * 60)
print("ALL MONTHS COMBINED")
print("=" * 60)

print(
    "Rows:",
    len(df_2025)
)

print(
    "Columns:",
    len(df_2025.columns)
)


# ============================================================
# 6. EXPECTED COLUMNS
# ============================================================

expected_columns = [
    "date",
    "city",
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

if list(df_2025.columns) != expected_columns:

    raise ValueError(
        "Column structure does not match expected schema."
    )

print(
    "Column structure: PASS"
)


# ============================================================
# 7. YEAR CHECK
# ============================================================

years = sorted(
    df_2025["date"]
    .dt.year
    .unique()
)

print("\nYears found:", years)

if years != [YEAR]:

    raise ValueError(
        f"Unexpected years found: {years}"
    )

print(
    "Year check: PASS"
)


# ============================================================
# 8. DATE RANGE
# ============================================================

min_date = df_2025["date"].min()

max_date = df_2025["date"].max()

print("\nDate range:")
print(
    min_date,
    "→",
    max_date
)

if (
    min_date != pd.Timestamp("2025-01-01")
    or
    max_date != pd.Timestamp("2025-12-31")
):

    raise ValueError(
        "Incorrect date range."
    )

print(
    "Date range: PASS"
)


# ============================================================
# 9. TOTAL ROW COUNT
# ============================================================

expected_rows = 9 * 365

actual_rows = len(df_2025)

print("\nRow count:")
print(
    "Expected:",
    expected_rows
)

print(
    "Actual:",
    actual_rows
)

if actual_rows != expected_rows:

    raise ValueError(
        f"Expected {expected_rows} rows "
        f"but found {actual_rows}."
    )

print(
    "Row count: PASS"
)


# ============================================================
# 10. CITY CHECK
# ============================================================

actual_cities = set(
    df_2025["city"].unique()
)

print("\nCities found:")
print(
    sorted(actual_cities)
)

print(
    "Number of cities:",
    len(actual_cities)
)

if actual_cities != expected_cities:

    print(
        "Missing cities:",
        expected_cities - actual_cities
    )

    print(
        "Unexpected cities:",
        actual_cities - expected_cities
    )

    raise ValueError(
        "City validation failed."
    )

print(
    "City check: PASS"
)


# ============================================================
# 11. ROWS PER CITY
# ============================================================

rows_per_city = (
    df_2025
    .groupby("city")["date"]
    .count()
    .sort_index()
)

print("\nRows per city:")
print(
    rows_per_city
)

if not (rows_per_city == 365).all():

    raise ValueError(
        "At least one city does not have 365 rows."
    )

print(
    "Rows per city: PASS"
)


# ============================================================
# 12. UNIQUE DATES
# ============================================================

unique_dates = (
    df_2025["date"].nunique()
)

print("\nUnique dates:", unique_dates)

if unique_dates != 365:

    raise ValueError(
        f"Expected 365 unique dates, "
        f"found {unique_dates}."
    )

print(
    "Unique dates: PASS"
)


# ============================================================
# 13. CITY-DATE GRAIN
# ============================================================

duplicate_city_date = (
    df_2025
    .duplicated(
        subset=["city", "date"]
    )
    .sum()
)

print(
    "\nDuplicate city-date combinations:",
    duplicate_city_date
)

if duplicate_city_date != 0:

    raise ValueError(
        "Duplicate city-date records found."
    )

print(
    "City-date grain: PASS"
)


# ============================================================
# 14. MISSING VALUES
# ============================================================

missing_values = (
    df_2025
    .isna()
    .sum()
)

print("\nMissing values:")
print(
    missing_values
)

if missing_values.sum() != 0:

    raise ValueError(
        "Missing values detected."
    )

print(
    "Missing values: PASS"
)


# ============================================================
# 15. FULL DUPLICATES
# ============================================================

full_duplicates = (
    df_2025
    .duplicated()
    .sum()
)

print(
    "\nFull duplicate rows:",
    full_duplicates
)

if full_duplicates != 0:

    raise ValueError(
        "Full duplicate rows found."
    )

print(
    "Full duplicates: PASS"
)


# ============================================================
# 16. TEMPERATURE LOGIC
# ============================================================

invalid_temperature = (
    (df_2025["temp_min"] > df_2025["temp_avg"])
    |
    (df_2025["temp_avg"] > df_2025["temp_max"])
)

invalid_temp_count = (
    invalid_temperature.sum()
)

print(
    "\nInvalid temperature records:",
    invalid_temp_count
)

if invalid_temp_count != 0:

    raise ValueError(
        "Invalid temperature relationships found."
    )

print(
    "Temperature logic: PASS"
)


# ============================================================
# 17. PRECIPITATION
# ============================================================

negative_precipitation = (
    df_2025["precipitation"] < 0
).sum()

print(
    "\nNegative precipitation:",
    negative_precipitation
)

if negative_precipitation != 0:

    raise ValueError(
        "Negative precipitation detected."
    )

print(
    "Precipitation: PASS"
)


# ============================================================
# 18. HUMIDITY
# ============================================================

invalid_humidity = (
    (df_2025["humidity"] < 0)
    |
    (df_2025["humidity"] > 100)
).sum()

print(
    "\nInvalid humidity:",
    invalid_humidity
)

if invalid_humidity != 0:

    raise ValueError(
        "Invalid humidity values detected."
    )

print(
    "Humidity: PASS"
)


# ============================================================
# 19. WIND SPEED
# ============================================================

negative_wind = (
    df_2025["wind_speed"] < 0
).sum()

print(
    "\nNegative wind speed:",
    negative_wind
)

if negative_wind != 0:

    raise ValueError(
        "Negative wind speed detected."
    )

print(
    "Wind speed: PASS"
)


# ============================================================
# 20. COMPLETE CITY-DATE COVERAGE
# ============================================================

expected_dates = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="D"
)

coverage_errors = []

print("\n")
print("=" * 60)
print("CITY-DATE COVERAGE")
print("=" * 60)

for city in sorted(expected_cities):

    city_dates = (
        df_2025.loc[
            df_2025["city"] == city,
            "date"
        ]
        .drop_duplicates()
        .sort_values()
    )

    missing_dates = (
        expected_dates
        .difference(city_dates)
    )

    extra_dates = (
        city_dates[
            ~city_dates.isin(
                expected_dates
            )
        ]
    )

    print(
        f"{city}: "
        f"{len(city_dates)} dates | "
        f"Missing: {len(missing_dates)}"
    )

    if len(missing_dates) > 0:

        coverage_errors.append(
            (
                city,
                "Missing",
                list(missing_dates)
            )
        )

    if len(extra_dates) > 0:

        coverage_errors.append(
            (
                city,
                "Unexpected",
                list(extra_dates)
            )
        )


if coverage_errors:

    print("\nCoverage errors:")

    for error in coverage_errors:
        print(error)

    raise ValueError(
        "City-date coverage validation failed."
    )

print(
    "\nComplete city-date coverage: PASS"
)


# ============================================================
# 21. SORT FINAL DATASET
# ============================================================

df_2025 = (
    df_2025
    .sort_values(
        ["date", "city"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 22. SAVE FINAL DATASET
# ============================================================

df_2025.to_csv(
    final_csv,
    index=False
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("FINAL 2025 DATASET CREATED")
print("=" * 60)

print(
    "File:",
    final_csv
)

print(
    "Rows:",
    len(df_2025)
)

print(
    "Columns:",
    len(df_2025.columns)
)

print(
    "Cities:",
    df_2025["city"].nunique()
)

print(
    "Dates:",
    df_2025["date"].nunique()
)

print("\n")
print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print("Rows:                 PASS")
print("Columns:              PASS")
print("Year:                 PASS")
print("Date range:           PASS")
print("Cities:               PASS")
print("Rows per city:        PASS")
print("Unique dates:         PASS")
print("City-date grain:      PASS")
print("Missing values:       PASS")
print("Full duplicates:      PASS")
print("Temperature logic:    PASS")
print("Precipitation:        PASS")
print("Humidity:             PASS")
print("Wind speed:           PASS")
print("Date coverage:        PASS")

print("=" * 60)
print("2025 DATASET STATUS: SUCCESS")
print("=" * 60)

LOADING 2025 MONTHLY CHECKPOINTS

Loading: 2025-01
Rows loaded: 279

Loading: 2025-02
Rows loaded: 252

Loading: 2025-03
Rows loaded: 279

Loading: 2025-04
Rows loaded: 270

Loading: 2025-05
Rows loaded: 279

Loading: 2025-06
Rows loaded: 270

Loading: 2025-07
Rows loaded: 279

Loading: 2025-08
Rows loaded: 279

Loading: 2025-09
Rows loaded: 270

Loading: 2025-10
Rows loaded: 279

Loading: 2025-11
Rows loaded: 270

Loading: 2025-12
Rows loaded: 279


ALL MONTHS COMBINED
Rows: 3285
Columns: 8
Column structure: PASS

Years found: [np.int32(2025)]
Year check: PASS

Date range:
2025-01-01 00:00:00 → 2025-12-31 00:00:00
Date range: PASS

Row count:
Expected: 3285
Actual: 3285
Row count: PASS

Cities found:
['Amman', 'Cairo', 'Istanbul', 'Khartoum', 'London', 'Munich', 'Riyadh', 'Salalah', 'Zurich']
Number of cities: 9
City check: PASS

Rows per city:
city
Amman       365
Cairo       365
Istanbul    365
Khartoum    365
London      365
Munich      365
Riyadh      365
Salalah     365
Zurich   

In [ ]:
# ============================================================
# EDA - PART 1
# Statistical & Distribution Overview
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Basic dataset information
# ------------------------------------------------------------

print("=" * 60)
print("EDA - DATASET OVERVIEW")
print("=" * 60)

print("Shape:", df_2025.shape)

print("\nData types:")
print(df_2025.dtypes)

print("\nMemory usage:")
print(df_2025.memory_usage(deep=True).sum(), "bytes")


# ------------------------------------------------------------
# 2. Statistical summary
# ------------------------------------------------------------

numeric_columns = [
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

print("\n")
print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)

print(
    df_2025[numeric_columns].describe()
)


# ------------------------------------------------------------
# 3. City-level statistics
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CITY-LEVEL TEMPERATURE STATISTICS")
print("=" * 60)

city_temperature = (
    df_2025
    .groupby("city")
    .agg(
        avg_temperature=("temp_avg", "mean"),
        max_temperature=("temp_max", "max"),
        min_temperature=("temp_min", "min")
    )
    .sort_values(
        "avg_temperature",
        ascending=False
    )
)

print(city_temperature)


# ------------------------------------------------------------
# 4. City-level precipitation
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CITY-LEVEL PRECIPITATION")
print("=" * 60)

city_precipitation = (
    df_2025
    .groupby("city")
    .agg(
        total_precipitation=("precipitation", "sum"),
        average_daily_precipitation=("precipitation", "mean"),
        rainy_days=(
            "precipitation",
            lambda x: (x > 0).sum()
        )
    )
    .sort_values(
        "total_precipitation",
        ascending=False
    )
)

print(city_precipitation)


# ------------------------------------------------------------
# 5. City-level humidity and wind
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CITY-LEVEL HUMIDITY & WIND")
print("=" * 60)

city_environment = (
    df_2025
    .groupby("city")
    .agg(
        average_humidity=("humidity", "mean"),
        max_humidity=("humidity", "max"),
        average_wind_speed=("wind_speed", "mean"),
        max_wind_speed=("wind_speed", "max")
    )
    .sort_values(
        "average_humidity",
        ascending=False
    )
)

print(city_environment)


# ------------------------------------------------------------
# 6. Correlation matrix
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("CORRELATION MATRIX")
print("=" * 60)

correlation = (
    df_2025[numeric_columns]
    .corr()
)

print(correlation)


# ------------------------------------------------------------
# 7. Extreme temperature records
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("TOP 10 HOTTEST DAYS")
print("=" * 60)

hottest_days = (
    df_2025
    .sort_values(
        "temp_max",
        ascending=False
    )
    [
        [
            "date",
            "city",
            "temp_avg",
            "temp_max",
            "temp_min"
        ]
    ]
    .head(10)
)

print(hottest_days)


print("\n")
print("=" * 60)
print("TOP 10 COLDEST DAYS")
print("=" * 60)

coldest_days = (
    df_2025
    .sort_values(
        "temp_min",
        ascending=True
    )
    [
        [
            "date",
            "city",
            "temp_avg",
            "temp_max",
            "temp_min"
        ]
    ]
    .head(10)
)

print(coldest_days)


# ------------------------------------------------------------
# 8. Highest precipitation days
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("TOP 10 RAINIEST DAYS")
print("=" * 60)

rainiest_days = (
    df_2025
    .sort_values(
        "precipitation",
        ascending=False
    )
    [
        [
            "date",
            "city",
            "precipitation"
        ]
    ]
    .head(10)
)

print(rainiest_days)


# ------------------------------------------------------------
# 9. Highest wind days
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("TOP 10 WINDIEST DAYS")
print("=" * 60)

windiest_days = (
    df_2025
    .sort_values(
        "wind_speed",
        ascending=False
    )
    [
        [
            "date",
            "city",
            "wind_speed"
        ]
    ]
    .head(10)
)

print(windiest_days)

print("\n")
print("=" * 60)
print("EDA PART 1 COMPLETED")
print("=" * 60)

EDA - DATASET OVERVIEW
Shape: (3285, 8)

Data types:
date             datetime64[us]
city                        str
temp_avg                float64
temp_max                float64
temp_min                float64
precipitation           float64
humidity                  int64
wind_speed              float64
dtype: object

Memory usage:
365862 bytes


STATISTICAL SUMMARY
          temp_avg     temp_max     temp_min  precipitation     humidity  \
count  3285.000000  3285.000000  3285.000000    3285.000000  3285.000000   
mean     19.707549    24.514581    15.251446       1.159756    56.631659   
std      10.004478    10.967675     9.020915       3.576252    25.148901   
min      -4.500000    -2.200000    -8.100000       0.000000     6.000000   
25%      11.900000    16.100000     8.500000       0.000000    37.000000   
50%      20.600000    25.900000    15.600000       0.000000    61.000000   
75%      27.200000    32.800000    22.700000       0.200000    78.000000   
max      40.400000 

In [ ]:
# ============================================================
# EDA - PART 1B
# Compact Summary
# ============================================================

print("=" * 60)
print("COMPACT EDA SUMMARY")
print("=" * 60)

print("\nDataset:")
print("Rows:", len(df_2025))
print("Cities:", df_2025["city"].nunique())
print("Date range:", df_2025["date"].min(), "to", df_2025["date"].max())

print("\n" + "=" * 60)
print("NUMERIC SUMMARY")
print("=" * 60)

summary = df_2025[
    [
        "temp_avg",
        "temp_max",
        "temp_min",
        "precipitation",
        "humidity",
        "wind_speed"
    ]
].agg(["min", "mean", "max"])

print(summary.round(2))

print("\n" + "=" * 60)
print("AVERAGE TEMPERATURE BY CITY")
print("=" * 60)

city_temp = (
    df_2025
    .groupby("city")["temp_avg"]
    .mean()
    .sort_values(ascending=False)
)

print(city_temp.round(2))

print("\n" + "=" * 60)
print("TOTAL PRECIPITATION BY CITY")
print("=" * 60)

city_rain = (
    df_2025
    .groupby("city")["precipitation"]
    .sum()
    .sort_values(ascending=False)
)

print(city_rain.round(2))

print("\n" + "=" * 60)
print("AVERAGE HUMIDITY BY CITY")
print("=" * 60)

city_humidity = (
    df_2025
    .groupby("city")["humidity"]
    .mean()
    .sort_values(ascending=False)
)

print(city_humidity.round(2))

print("\n" + "=" * 60)
print("AVERAGE WIND SPEED BY CITY")
print("=" * 60)

city_wind = (
    df_2025
    .groupby("city")["wind_speed"]
    .mean()
    .sort_values(ascending=False)
)

print(city_wind.round(2))

print("\n" + "=" * 60)
print("COMPACT SUMMARY COMPLETED")
print("=" * 60)

COMPACT EDA SUMMARY

Dataset:
Rows: 3285
Cities: 9
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00

NUMERIC SUMMARY
      temp_avg  temp_max  temp_min  precipitation  humidity  wind_speed
min      -4.50     -2.20     -8.10           0.00      6.00        1.00
mean     19.71     24.51     15.25           1.16     56.63        8.75
max      40.40     46.20     36.90          41.80     99.00       34.60

AVERAGE TEMPERATURE BY CITY
city
Khartoum    30.90
Riyadh      28.07
Salalah     26.09
Cairo       24.03
Amman       19.28
Istanbul    16.13
London      12.42
Zurich      10.38
Munich      10.08
Name: temp_avg, dtype: float64

TOTAL PRECIPITATION BY CITY
city
Zurich      1267.5
Munich       840.7
Istanbul     644.1
London       613.4
Salalah      153.8
Amman        130.3
Khartoum      77.3
Riyadh        73.9
Cairo          8.8
Name: precipitation, dtype: float64

AVERAGE HUMIDITY BY CITY
city
Zurich      81.24
Munich      75.37
London      73.56
Istanbul    69.52
Salalah     65.02


In [ ]:
# ============================================================
# EDA - PART 2
# Seasonality Analysis
# ============================================================

print("=" * 60)
print("EDA - SEASONAL ANALYSIS")
print("=" * 60)

# Create month information
df_2025["month"] = df_2025["date"].dt.month
df_2025["month_name"] = df_2025["date"].dt.month_name()

# Monthly temperature
monthly_temperature = (
    df_2025
    .groupby("month")
    .agg(
        avg_temp=("temp_avg", "mean"),
        max_temp=("temp_max", "max"),
        min_temp=("temp_min", "min")
    )
)

print("\n" + "=" * 60)
print("MONTHLY TEMPERATURE")
print("=" * 60)

print(monthly_temperature.round(2))


# Monthly precipitation
monthly_rain = (
    df_2025
    .groupby("month")
    .agg(
        total_precipitation=("precipitation", "sum"),
        rainy_days=("precipitation", lambda x: (x > 0).sum())
    )
)

print("\n" + "=" * 60)
print("MONTHLY PRECIPITATION")
print("=" * 60)

print(monthly_rain.round(2))


# Monthly humidity
monthly_humidity = (
    df_2025
    .groupby("month")["humidity"]
    .mean()
)

print("\n" + "=" * 60)
print("MONTHLY HUMIDITY")
print("=" * 60)

print(monthly_humidity.round(2))


# Monthly wind
monthly_wind = (
    df_2025
    .groupby("month")["wind_speed"]
    .mean()
)

print("\n" + "=" * 60)
print("MONTHLY WIND SPEED")
print("=" * 60)

print(monthly_wind.round(2))


# Seasonal summary
df_2025["season"] = df_2025["month"].map({
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Summer",
    10: "Autumn",
    11: "Autumn"
})

seasonal_summary = (
    df_2025
    .groupby("season")
    .agg(
        avg_temperature=("temp_avg", "mean"),
        max_temperature=("temp_max", "max"),
        min_temperature=("temp_min", "min"),
        precipitation=("precipitation", "sum"),
        humidity=("humidity", "mean"),
        wind_speed=("wind_speed", "mean")
    )
)

print("\n" + "=" * 60)
print("SEASONAL SUMMARY")
print("=" * 60)

print(seasonal_summary.round(2))

print("\n" + "=" * 60)
print("EDA PART 2 COMPLETED")
print("=" * 60)

EDA - SEASONAL ANALYSIS

MONTHLY TEMPERATURE
       avg_temp  max_temp  min_temp
month                              
1         11.52      36.0      -7.9
2         11.47      37.5      -6.3
3         16.03      43.1      -3.3
4         19.88      44.3      -1.9
5         23.20      44.5       4.0
6         26.82      45.5       9.0
7         26.93      46.2       9.8
8         26.30      45.2       8.3
9         23.79      43.0       7.2
10        20.37      40.6       2.0
11        16.95      38.8      -7.5
12        12.71      37.7      -8.1

MONTHLY PRECIPITATION
       total_precipitation  rainy_days
month                                 
1                    358.7          77
2                    260.0          82
3                    222.5          85
4                    150.3          48
5                    323.3          84
6                    237.4          61
7                    463.4         103
8                    380.8         100
9                    398.2         100

In [ ]:
# ============================================================
# EDA - PART 2B
# Compact Seasonal Summary
# ============================================================

print("=" * 60)
print("SEASONAL COMPACT SUMMARY")
print("=" * 60)

print("\nMONTHLY TEMPERATURE")
print(
    monthly_temperature.round(2).to_string()
)

print("\n" + "=" * 60)
print("MONTHLY PRECIPITATION")
print("=" * 60)

print(
    monthly_rain.round(2).to_string()
)

print("\n" + "=" * 60)
print("MONTHLY HUMIDITY")
print("=" * 60)

print(
    monthly_humidity.round(2).to_string()
)

print("\n" + "=" * 60)
print("MONTHLY WIND")
print("=" * 60)

print(
    monthly_wind.round(2).to_string()
)

print("\n" + "=" * 60)
print("SEASONAL SUMMARY")
print("=" * 60)

print(
    seasonal_summary.round(2).to_string()
)

print("\n" + "=" * 60)
print("EDA PART 2B COMPLETED")
print("=" * 60)

SEASONAL COMPACT SUMMARY

MONTHLY TEMPERATURE
       avg_temp  max_temp  min_temp
month                              
1         11.52      36.0      -7.9
2         11.47      37.5      -6.3
3         16.03      43.1      -3.3
4         19.88      44.3      -1.9
5         23.20      44.5       4.0
6         26.82      45.5       9.0
7         26.93      46.2       9.8
8         26.30      45.2       8.3
9         23.79      43.0       7.2
10        20.37      40.6       2.0
11        16.95      38.8      -7.5
12        12.71      37.7      -8.1

MONTHLY PRECIPITATION
       total_precipitation  rainy_days
month                                 
1                    358.7          77
2                    260.0          82
3                    222.5          85
4                    150.3          48
5                    323.3          84
6                    237.4          61
7                    463.4         103
8                    380.8         100
9                    398.2         10

In [ ]:
# ============================================================
# EDA - PART 3
# Outlier & Data Quality Analysis
# ============================================================

print("=" * 60)
print("OUTLIER & DATA QUALITY ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# 1. IQR OUTLIER DETECTION
# ------------------------------------------------------------

numeric_columns = [
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

print("\n" + "=" * 60)
print("IQR OUTLIERS")
print("=" * 60)

for column in numeric_columns:

    Q1 = df_2025[column].quantile(0.25)
    Q3 = df_2025[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_2025[
        (df_2025[column] < lower_bound) |
        (df_2025[column] > upper_bound)
    ]

    print(f"\n{column}")
    print("Q1:", round(Q1, 2))
    print("Q3:", round(Q3, 2))
    print("IQR:", round(IQR, 2))
    print("Lower bound:", round(lower_bound, 2))
    print("Upper bound:", round(upper_bound, 2))
    print("Outlier count:", len(outliers))


# ------------------------------------------------------------
# 2. EXTREME TEMPERATURE DAYS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXTREME TEMPERATURE DAYS")
print("=" * 60)

hot_days = df_2025[
    df_2025["temp_max"] >= 45
][
    [
        "date",
        "city",
        "temp_avg",
        "temp_max",
        "temp_min"
    ]
].sort_values("temp_max", ascending=False)

print("\nDays with maximum temperature >= 45°C:")
print(hot_days.to_string(index=False))


cold_days = df_2025[
    df_2025["temp_min"] <= -5
][
    [
        "date",
        "city",
        "temp_avg",
        "temp_max",
        "temp_min"
    ]
].sort_values("temp_min")

print("\nDays with minimum temperature <= -5°C:")
print(cold_days.to_string(index=False))


# ------------------------------------------------------------
# 3. EXTREME PRECIPITATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXTREME PRECIPITATION")
print("=" * 60)

heavy_rain = df_2025[
    df_2025["precipitation"] >= 20
][
    [
        "date",
        "city",
        "precipitation"
    ]
].sort_values(
    "precipitation",
    ascending=False
)

print(
    heavy_rain.to_string(index=False)
)


# ------------------------------------------------------------
# 4. EXTREME WIND
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXTREME WIND")
print("=" * 60)

strong_wind = df_2025[
    df_2025["wind_speed"] >= 25
][
    [
        "date",
        "city",
        "wind_speed"
    ]
].sort_values(
    "wind_speed",
    ascending=False
)

print(
    strong_wind.to_string(index=False)
)


# ------------------------------------------------------------
# 5. BASIC DATA QUALITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print("Missing values:")
print(df_2025.isna().sum())

print("\nDuplicate rows:")
print(df_2025.duplicated().sum())

print("\nInvalid temperature relationships:")

invalid_temperature = df_2025[
    (df_2025["temp_min"] > df_2025["temp_avg"]) |
    (df_2025["temp_avg"] > df_2025["temp_max"])
]

print(len(invalid_temperature))

print("\nInvalid humidity:")
print(
    len(
        df_2025[
            (df_2025["humidity"] < 0) |
            (df_2025["humidity"] > 100)
        ]
    )
)

print("\nInvalid precipitation:")
print(
    len(
        df_2025[
            df_2025["precipitation"] < 0
        ]
    )
)

print("\nInvalid wind speed:")
print(
    len(
        df_2025[
            df_2025["wind_speed"] < 0
        ]
    )
)

print("\n" + "=" * 60)
print("EDA PART 3 COMPLETED")
print("=" * 60)

OUTLIER & DATA QUALITY ANALYSIS

IQR OUTLIERS

temp_avg
Q1: 11.9
Q3: 27.2
IQR: 15.3
Lower bound: -11.05
Upper bound: 50.15
Outlier count: 0

temp_max
Q1: 16.1
Q3: 32.8
IQR: 16.7
Lower bound: -8.95
Upper bound: 57.85
Outlier count: 0

temp_min
Q1: 8.5
Q3: 22.7
IQR: 14.2
Lower bound: -12.8
Upper bound: 44.0
Outlier count: 0

precipitation
Q1: 0.0
Q3: 0.2
IQR: 0.2
Lower bound: -0.3
Upper bound: 0.5
Outlier count: 687

humidity
Q1: 37.0
Q3: 78.0
IQR: 41.0
Lower bound: -24.5
Upper bound: 139.5
Outlier count: 0

wind_speed
Q1: 6.1
Q3: 10.6
IQR: 4.5
Lower bound: -0.65
Upper bound: 17.35
Outlier count: 122

EXTREME TEMPERATURE DAYS

Days with maximum temperature >= 45°C:
      date     city  temp_avg  temp_max  temp_min
2025-07-30   Riyadh      39.9      46.2      33.3
2025-07-29   Riyadh      40.1      45.6      33.3
2025-07-28   Riyadh      40.1      45.6      33.5
2025-06-16   Riyadh      39.3      45.5      32.5
2025-06-13 Khartoum      38.6      45.4      31.0
2025-08-11   Riyadh      40.

In [ ]:
# ============================================================
# EDA - PART 4
# Correlation & Relationships
# ============================================================

print("=" * 60)
print("CORRELATION & RELATIONSHIP ANALYSIS")
print("=" * 60)

numeric_columns = [
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "humidity",
    "wind_speed"
]

correlation = df_2025[numeric_columns].corr()

print("\nCorrelation Matrix:")
print(correlation.round(2).to_string())

print("\n" + "=" * 60)
print("STRONGEST CORRELATIONS")
print("=" * 60)

# Keep only unique pairs
pairs = []

for i in range(len(numeric_columns)):
    for j in range(i + 1, len(numeric_columns)):

        col1 = numeric_columns[i]
        col2 = numeric_columns[j]

        value = correlation.loc[col1, col2]

        pairs.append(
            {
                "variable_1": col1,
                "variable_2": col2,
                "correlation": value
            }
        )

correlation_pairs = pd.DataFrame(pairs)

correlation_pairs["abs_correlation"] = (
    correlation_pairs["correlation"].abs()
)

correlation_pairs = (
    correlation_pairs
    .sort_values(
        "abs_correlation",
        ascending=False
    )
    .drop(columns="abs_correlation")
)

print(
    correlation_pairs
    .round(2)
    .to_string(index=False)
)

print("\n" + "=" * 60)
print("CITY TEMPERATURE RANGE")
print("=" * 60)

temperature_range = (
    df_2025
    .groupby("city")
    .agg(
        avg_temp=("temp_avg", "mean"),
        avg_max=("temp_max", "mean"),
        avg_min=("temp_min", "mean")
    )
)

temperature_range["daily_temperature_range"] = (
    temperature_range["avg_max"]
    - temperature_range["avg_min"]
)

print(
    temperature_range
    .sort_values(
        "daily_temperature_range",
        ascending=False
    )
    .round(2)
    .to_string()
)

print("\n" + "=" * 60)
print("EDA PART 4 COMPLETED")
print("=" * 60)

CORRELATION & RELATIONSHIP ANALYSIS

Correlation Matrix:
               temp_avg  temp_max  temp_min  precipitation  humidity  wind_speed
temp_avg           1.00      0.99      0.98          -0.22     -0.73        0.01
temp_max           0.99      1.00      0.95          -0.25     -0.76       -0.03
temp_min           0.98      0.95      1.00          -0.17     -0.65        0.05
precipitation     -0.22     -0.25     -0.17           1.00      0.34        0.14
humidity          -0.73     -0.76     -0.65           0.34      1.00       -0.04
wind_speed         0.01     -0.03      0.05           0.14     -0.04        1.00

STRONGEST CORRELATIONS
   variable_1    variable_2  correlation
     temp_avg      temp_max         0.99
     temp_avg      temp_min         0.98
     temp_max      temp_min         0.95
     temp_max      humidity        -0.76
     temp_avg      humidity        -0.73
     temp_min      humidity        -0.65
precipitation      humidity         0.34
     temp_max precipitat

In [ ]:
from pathlib import Path
import sys

# Make the project root importable from a notebook session
for candidate in [Path.cwd(), Path.cwd().parent]:
    candidate = candidate.resolve()
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

try:
    from src.eda import run_eda
except ModuleNotFoundError:
    # Fallback for notebook environments launched from a subfolder
    repo_root = next(
        (p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists()),
        None
    )
    if repo_root is not None and str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root.resolve()))
    from src.eda import run_eda

eda_results = run_eda(
    df_2025,
    year=2025
)

print(eda_results["overview"])

{'rows': 3285, 'columns': 11, 'cities': 9, 'start_date': Timestamp('2025-01-01 00:00:00'), 'end_date': Timestamp('2025-12-31 00:00:00'), 'memory_bytes': np.int64(740838)}


In [ ]:
print(df_2025.columns.tolist())

['date', 'city', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'month', 'month_name', 'season']


In [ ]:
import importlib
import src.transformation

importlib.reload(src.transformation)

from src.transformation import prepare_dates

df_transformed = prepare_dates(df_2025)

print(df_transformed.dtypes)

date             datetime64[us]
city                        str
temp_avg                float64
temp_max                float64
temp_min                float64
precipitation           float64
humidity                  int64
wind_speed              float64
dtype: object


In [ ]:
from src.transformation import rename_columns

raw_test = pd.DataFrame({
    "time": ["2025-01-01"],
    "temperature_2m_mean": [20.5],
    "temperature_2m_max": [25.0],
    "temperature_2m_min": [15.0],
    "precipitation_sum": [1.2],
    "relative_humidity_2m_mean": [60],
    "wind_speed_10m_mean": [8.5],
    "city": ["Cairo"],
})

renamed_test = rename_columns(raw_test)

print(renamed_test.columns.tolist())
print(renamed_test)

['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city']
         date  temp_avg  temp_max  temp_min  precipitation  humidity  \
0  2025-01-01      20.5      25.0      15.0            1.2        60   

   wind_speed   city  
0         8.5  Cairo  


In [ ]:
for name in ["cities", "locations_df", "weather_raw"]:
    print(name, "->", "defined" if name in globals() else "MISSING")

cities -> MISSING
locations_df -> MISSING
weather_raw -> MISSING


In [ ]:
from pathlib import Path
import sys

repo_root = next(
    (
        p.resolve()
        for p in [Path.cwd(), Path.cwd().parent]
        if p.exists() and (p / "src").exists()
    ),
    None,
)

if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.geocoding import get_locations

locations_df = get_locations()

print(locations_df)
print(locations_df.shape)

       city              country  latitude  longitude         timezone
0    Riyadh         Saudi Arabia  24.68773   46.72185      Asia/Riyadh
1   Salalah                 Oman  17.01505   54.09237      Asia/Muscat
2     Amman               Jordan  31.95522   35.94503       Asia/Amman
3  Istanbul  Republic of Türkiye  41.01384   28.94966  Europe/Istanbul
4    London       United Kingdom  51.50853   -0.12574    Europe/London
5    Munich              Germany  48.13743   11.57549    Europe/Berlin
6    Zurich          Switzerland  47.36667    8.55000    Europe/Zurich
7     Cairo                Egypt  30.06263   31.24967     Africa/Cairo
8  Khartoum                Sudan  15.55177   32.53241  Africa/Khartoum
(9, 5)


In [ ]:
import os
print(os.getcwd())
print(os.listdir("src"))

c:\Users\DELL\api_data_pipeline
['eda.py', 'geocoding', 'transformation.py', '__pycache__']


In [ ]:
import os
print(os.path.isdir("src/geocoding"))
print(os.path.isfile("src/geocoding"))

False
True


In [ ]:
import os
os.rename("src/geocoding", "src/geocoding.py")
print(os.listdir("src"))

['eda.py', 'geocoding.py', 'transformation.py', '__pycache__']


In [ ]:
import importlib
import src.geocoding

importlib.reload(src.geocoding)

from src.geocoding import get_locations

locations_df = get_locations()

print(locations_df)
print("\nShape:", locations_df.shape)

       city              country  latitude  longitude         timezone
0    Riyadh         Saudi Arabia  24.68773   46.72185      Asia/Riyadh
1   Salalah                 Oman  17.01505   54.09237      Asia/Muscat
2     Amman               Jordan  31.95522   35.94503       Asia/Amman
3  Istanbul  Republic of Türkiye  41.01384   28.94966  Europe/Istanbul
4    London       United Kingdom  51.50853   -0.12574    Europe/London
5    Munich              Germany  48.13743   11.57549    Europe/Berlin
6    Zurich          Switzerland  47.36667    8.55000    Europe/Zurich
7     Cairo                Egypt  30.06263   31.24967     Africa/Cairo
8  Khartoum                Sudan  15.55177   32.53241  Africa/Khartoum

Shape: (9, 5)


In [ ]:
from src.transformation import build_dim_city

dim_city = build_dim_city(locations_df)

print(dim_city)
print("\nShape:", dim_city.shape)

   city_key      city              country  latitude  longitude
0         1    Riyadh         Saudi Arabia  24.68773   46.72185
1         2   Salalah                 Oman  17.01505   54.09237
2         3     Amman               Jordan  31.95522   35.94503
3         4  Istanbul  Republic of Türkiye  41.01384   28.94966
4         5    London       United Kingdom  51.50853   -0.12574
5         6    Munich              Germany  48.13743   11.57549
6         7    Zurich          Switzerland  47.36667    8.55000
7         8     Cairo                Egypt  30.06263   31.24967
8         9  Khartoum                Sudan  15.55177   32.53241

Shape: (9, 5)


In [ ]:
import importlib
import src.transformation

importlib.reload(src.transformation)

from src.transformation import attach_city_key

weather_with_city_key = attach_city_key(
    df_transformed,
    dim_city
)

print(weather_with_city_key.head())
print("\nColumns:")
print(weather_with_city_key.columns.tolist())

print("\nShape:")
print(weather_with_city_key.shape)

        date  temp_avg  temp_max  temp_min  precipitation  humidity  \
0 2025-01-01       8.1      12.6       5.1            0.0        78   
1 2025-01-01      14.7      20.0       9.9            0.0        64   
2 2025-01-01       5.9       9.0       3.0            0.0        90   
3 2025-01-01      21.9      28.7      15.8            0.0        27   
4 2025-01-01       9.0      11.8       5.3           10.5        87   

   wind_speed  city_key  
0         2.9         3  
1         5.2         8  
2         3.3         4  
3        14.4         9  
4        21.8         5  

Columns:
['date', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city_key']

Shape:
(3285, 8)


In [ ]:
import importlib
import src.transformation

importlib.reload(src.transformation)

from src.transformation import build_fact_weather

fact_weather = build_fact_weather(
    df_2025,
    dim_city,
    dim_date
)

print(fact_weather.head())
print("\nColumns:")
print(fact_weather.columns.tolist())
print("\nShape:")
print(fact_weather.shape)

   temp_avg  temp_max  temp_min  precipitation  humidity  wind_speed  \
0       8.1      12.6       5.1            0.0        78         2.9   
1      14.7      20.0       9.9            0.0        64         5.2   
2       5.9       9.0       3.0            0.0        90         3.3   
3      21.9      28.7      15.8            0.0        27        14.4   
4       9.0      11.8       5.3           10.5        87        21.8   

   city_key  date_key  
0         3  20250101  
1         8  20250101  
2         4  20250101  
3         9  20250101  
4         5  20250101  

Columns:
['temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city_key', 'date_key']

Shape:
(3285, 8)


In [ ]:
# Validate fact table grain

print("Rows:", len(fact_weather))

print(
    "Unique city-date combinations:",
    fact_weather[["city_key", "date_key"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate city-date rows:",
    fact_weather.duplicated(
        subset=["city_key", "date_key"]
    ).sum()
)

print(
    "Missing values:",
    fact_weather.isna().sum().sum()
)

Rows: 3285
Unique city-date combinations: 3285
Duplicate city-date rows: 0
Missing values: 0


In [73]:
print(fact_weather.columns.tolist())
print(fact_weather.shape)
print(fact_weather["weather_id"].head())

['weather_id', 'temp_avg', 'temp_max', 'temp_min', 'precipitation', 'humidity', 'wind_speed', 'city_key', 'date_key']
(3285, 9)
0    1
1    2
2    3
3    4
4    5
Name: weather_id, dtype: int64


In [75]:
from pathlib import Path

backup_dir = Path("data/final/backup")
backup_dir.mkdir(parents=True, exist_ok=True)

fact_weather.to_csv(
    backup_dir / "fact_weather.csv",
    index=False
)

dim_city.to_csv(
    backup_dir / "dim_city.csv",
    index=False
)

dim_date.to_csv(
    backup_dir / "dim_date.csv",
    index=False
)

print("Backup files saved:")
print(backup_dir / "fact_weather.csv")
print(backup_dir / "dim_city.csv")
print(backup_dir / "dim_date.csv")

Backup files saved:
data\final\backup\fact_weather.csv
data\final\backup\dim_city.csv
data\final\backup\dim_date.csv


In [1]:
dim_city

NameError: name 'dim_city' is not defined

In [2]:
from src.transformation import build_dim_city

dim_city = build_dim_city(locations_df)

dim_city

NameError: name 'locations_df' is not defined

In [3]:
from src.geocoding import get_locations

cities = [
    "Riyadh",
    "Salalah",
    "Amman",
    "Istanbul",
    "London",
    "Munich",
    "Zurich",
    "Cairo",
    "Khartoum"
]

locations_df = get_locations(cities)

locations_df

,city,country,latitude,longitude,timezone
0,Riyadh,Saudi Arabia,24.68773,46.72185,Asia/Riyadh
1,Salalah,Oman,17.01505,54.09237,Asia/Muscat
2,Amman,Jordan,31.95522,35.94503,Asia/Amman
3,Istanbul,Republic of Türkiye,41.01384,28.94966,Europe/Istanbul
4,London,United Kingdom,51.50853,-0.12574,Europe/London
5,Munich,Germany,48.13743,11.57549,Europe/Berlin
6,Zurich,Switzerland,47.36667,8.55000,Europe/Zurich
7,Cairo,Egypt,30.06263,31.24967,Africa/Cairo
8,Khartoum,Sudan,15.55177,32.53241,Africa/Khartoum


In [4]:
from src.transformation import build_dim_city

dim_city = build_dim_city(locations_df)

dim_city

,city_key,city,country,latitude,longitude
0,1,Riyadh,Saudi Arabia,24.68773,46.72185
1,2,Salalah,Oman,17.01505,54.09237
2,3,Amman,Jordan,31.95522,35.94503
3,4,Istanbul,Republic of Türkiye,41.01384,28.94966
4,5,London,United Kingdom,51.50853,-0.12574
5,6,Munich,Germany,48.13743,11.57549
6,7,Zurich,Switzerland,47.36667,8.55000
7,8,Cairo,Egypt,30.06263,31.24967
8,9,Khartoum,Sudan,15.55177,32.53241


In [7]:
import sys

!{sys.executable} -m pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/17.7 MB 2.6 MB/s eta 0:00:07
   -- ------------------------------------- 1.0/17.7 MB 3.5 MB/s eta 0:00:05
   ----- ---------------------------------- 2.4/17.7 MB 4.2 MB/s eta 0:00:04
   ----- ---------------------------------- 2.6/17.7 MB 4.2 MB/s eta 0:00:04
   -------- ------------------------------- 3.9/17.7 MB 3.8 MB/s eta 0:00:04
   ---------- ----------------------------- 4.7/17.7 MB 3.9 MB/s eta 0:00:04
   ------------- -------------------------- 5.8/17.7 MB 4.0 MB/s eta 0:00:03
   --------------- ------------------------ 6.8/17.7 MB 4.1 MB/s eta 0:00:03
   ----------------- ---------------------- 7.9/17.7 MB 4.2 MB/s eta 0:00:03
   -------------------- ------------------- 8.9/17.7 MB 4.3 MB/s eta 0:00:03
   ---------------------- ----------------- 10.0/17.7 MB 4.3 MB/s eta 0:00:02
   ---------

In [1]:
import mysql.connector

print("MySQL connector is ready")

MySQL connector is ready


In [1]:
import sys
sys.path.insert(0, "src")

from database import (
    get_connection,
    run_query,
    load_dim_city,
    load_dim_date,
    load_fact_weather
)

In [5]:
import sys

sys.path.insert(0, "src")

from database import (
    get_connection,
    run_query,
    load_dim_city,
    load_dim_date,
    load_fact_weather
)

print("Database module loaded successfully!")

Database module loaded successfully!


In [12]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_file = Path.cwd() / ".env"

print("Exists:", env_file.exists())

load_dotenv(env_file, override=True)

print("HOST:", repr(os.getenv("DB_HOST")))
print("PORT:", repr(os.getenv("DB_PORT")))
print("USER:", repr(os.getenv("DB_USER")))
print("NAME:", repr(os.getenv("DB_NAME")))

Exists: True
HOST: '127.0.0.1'
PORT: '3306'
USER: 'root'
NAME: 'weather_warehouse'


In [13]:
import importlib
import database

importlib.reload(database)

conn = database.get_connection()

print("Connected successfully!")

conn.close()

Connected successfully!


In [15]:
from src.geocoding import get_locations
from src.transformation import build_dim_city, build_fact_weather
import pandas as pd

# Cities
cities = [
    "Riyadh",
    "Salalah",
    "Amman",
    "Istanbul",
    "London",
    "Munich",
    "Zurich",
    "Cairo",
    "Khartoum"
]

# Locations → dim_city
locations_df = get_locations(cities)
dim_city = build_dim_city(locations_df)

# Load 2025 data
df_2025 = pd.read_csv("data/final/weather_2025.csv")

# dim_date
dim_date = df_2025[["date"]].copy()
dim_date["date"] = pd.to_datetime(dim_date["date"])
dim_date = dim_date.drop_duplicates().sort_values("date")
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date = dim_date[[
    "date_key", "date", "year", "month", "month_name"
]]

# fact_weather
fact_weather = build_fact_weather(
    df_2025,
    dim_city,
    dim_date
)

print("dim_city:", len(dim_city))
print("dim_date:", len(dim_date))
print("fact_weather:", len(fact_weather))

dim_city: 9
dim_date: 365
fact_weather: 3285


In [16]:
from database import load_dim_city, load_dim_date, load_fact_weather

load_dim_city(dim_city)
load_dim_date(dim_date)
load_fact_weather(fact_weather)

print("🎉 ALL DATA LOADED SUCCESSFULLY!")

🎉 ALL DATA LOADED SUCCESSFULLY!


In [17]:
from database import run_query

result = run_query("""
    SELECT *
    FROM fact_weather
    LIMIT 10
""")

result

[{'weather_id': 1,
  'city_key': 3,
  'date_key': 20250101,
  'temp_avg': Decimal('8.10'),
  'temp_max': Decimal('12.60'),
  'temp_min': Decimal('5.10'),
  'precipitation': Decimal('0.00'),
  'humidity': 78,
  'wind_speed': Decimal('2.90')},
 {'weather_id': 2,
  'city_key': 8,
  'date_key': 20250101,
  'temp_avg': Decimal('14.70'),
  'temp_max': Decimal('20.00'),
  'temp_min': Decimal('9.90'),
  'precipitation': Decimal('0.00'),
  'humidity': 64,
  'wind_speed': Decimal('5.20')},
 {'weather_id': 3,
  'city_key': 4,
  'date_key': 20250101,
  'temp_avg': Decimal('5.90'),
  'temp_max': Decimal('9.00'),
  'temp_min': Decimal('3.00'),
  'precipitation': Decimal('0.00'),
  'humidity': 90,
  'wind_speed': Decimal('3.30')},
 {'weather_id': 4,
  'city_key': 9,
  'date_key': 20250101,
  'temp_avg': Decimal('21.90'),
  'temp_max': Decimal('28.70'),
  'temp_min': Decimal('15.80'),
  'precipitation': Decimal('0.00'),
  'humidity': 27,
  'wind_speed': Decimal('14.40')},
 {'weather_id': 5,
  'city_ke

In [18]:
df = run_query("""
    SELECT
        c.city,
        c.country,
        d.date,
        f.temp_avg,
        f.temp_max,
        f.temp_min,
        f.precipitation,
        f.humidity,
        f.wind_speed
    FROM fact_weather f
    JOIN dim_city c
        ON f.city_key = c.city_key
    JOIN dim_date d
        ON f.date_key = d.date_key
    LIMIT 10
""", as_dataframe=True)

df

c:\Users\DELL\api_data_pipeline\src\database.py:57: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,country,date,temp_avg,temp_max,temp_min,precipitation,humidity,wind_speed
0,Khartoum,Sudan,2025-01-01,21.9,28.7,15.8,0.0,27,14.4
1,Cairo,Egypt,2025-01-01,14.7,20.0,9.9,0.0,64,5.2
2,Zurich,Switzerland,2025-01-01,0.6,7.5,-3.3,0.0,87,4.6
3,Munich,Germany,2025-01-01,4.8,10.7,1.2,0.0,42,14.3
4,London,United Kingdom,2025-01-01,9.0,11.8,5.3,10.5,87,21.8
5,Istanbul,Republic of Türkiye,2025-01-01,5.9,9.0,3.0,0.0,90,3.3
6,Amman,Jordan,2025-01-01,8.1,12.6,5.1,0.0,78,2.9
7,Salalah,Oman,2025-01-01,21.9,26.2,17.7,0.0,68,6.1
8,Riyadh,Saudi Arabia,2025-01-01,13.8,17.6,9.7,0.0,34,8.6
9,Khartoum,Sudan,2025-01-02,22.6,29.6,16.3,0.0,29,13.0
